In [1]:
!pip install geopandas matplotlib
!pip install geodatasets
!pip install rioxarray
!pip install openpyxl
!pip install selenium webdriver-manager
!pip install fiona

!pip install netCDF4
!pip install seaborn
!pip install scipy
!pip install pygam
!pip install mplcursors

In [4]:

import copernicusmarine
from netCDF4 import Dataset, num2date
import pandas as pd
import matplotlib
import mplcursors
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import FixedLocator
from matplotlib.lines import Line2D
import os
import xarray as xr
import rioxarray as rxr
import seaborn as sns
import seaborn.objects as so
import numpy as np
import gc
from datetime import datetime
import glob
from scipy import stats
from scipy.stats import linregress
import geopandas as gpd
import geodatasets
from shapely.geometry import Point
from shapely.geometry import MultiPolygon
from shapely import wkb
import pandas as pd
import re
import time
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from scipy import stats
from statsmodels.tsa.seasonal import STL
import geopandas as gpd
import rioxarray
import fiona
from pygam import LinearGAM, s, f
import numpy as np

In [3]:
## makes all the text not italisized when regex latex is applied
plt.rcParams['mathtext.default'] = 'regular'

# Chapter 1
- Part 1: Spatial Mapping:

        - Use the stats I've already made to get maps of trend slope and other constituents over time. Make sure they are clean and ready for results.

- Part 2: Temporal Trajectories and Breakpoints:

        - Extract 2 spatial means time series: one for coastal and one for open ocean (make sure to only take points not in irish sea)
        - Use Seasonal Trende Decomposition using LEOSS to separate long term climate trends from repeating annual cycles
        - Get an overall trend and a seasonal trend
        - Overall trend tells you when darkening was worse/better
        - Seasonal trend can tell you if dates of bloom or winter storms are moving

- Part 3: Constiuent Attribution Matrix:

        - 2 different analyses on coastal and open ocean
        - 2 Options:
                - Option A: Correlated phase 1 slope maps of Kd490 and other constituents and see where they match
                - Option B: Fit a specialized GAM to isolated trend components derivedfrom STL decomp in Phase 2. Allows you to evaluate how long term baseline change in consituents drives long term Kd490, removing seasonal noise

## Part 1: Spatial Mapping

### Land Masses

In [50]:
## try to get this onto a map of Ireland using geopandas

url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IRL_shp.zip"
uk_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_GBR_shp.zip"


ireland = gpd.read_file(url, layer="gadm41_IRL_0")

uk = gpd.read_file(uk_url, layer="gadm41_GBR_0")
uk_regions = gpd.read_file(uk_url, layer="gadm41_GBR_1")
northern_ireland = uk_regions[uk_regions["NAME_1"] == "Northern Ireland"]
island = gpd.GeoDataFrame(pd.concat([ireland, northern_ireland], ignore_index=True), crs="EPSG:4326")

In [51]:
## adding in france for the large satellite plots
fra_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_FRA_shp.zip"
france = gpd.read_file(fra_url, layer="gadm41_FRA_0")

In [52]:
all_land = gpd.GeoDataFrame(pd.concat([ireland, uk_regions, france], ignore_index=True), crs="EPSG:4326")
landmass = all_land.clip([-15,48, -2, 59])

In [53]:
landmass.plot()

<Axes: >

### Kd490 Plots

In [54]:
file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/*.nc'))
kd_ds = xr.open_mfdataset(file_list, concat_dim='time', combine='nested')
## combine all the files into one, massive DataSet so that we can do comparisons over years
weekly_kd = kd_ds['KD490'].resample(time='1W').mean()
monthly_kd = kd_ds['KD490'].resample(time='1ME').mean()
seasonal_kd = kd_ds['KD490'].resample(time='QS-DEC').mean()
seasonal_kd['season'] = seasonal_kd['time'].dt.season

stat_trend_kd = xr.open_dataset('thesis_data/KD490_trend_with_stats.nc')  ## this is weekly trends

significant_kd = stat_trend_kd['pvalue'] <= 0.05  # 95% confidence
trend_sig_kd = stat_trend_kd['slope'].where(significant_kd)

seasonal_kd = xr.open_dataset('thesis_data/KD490_seasonal_trends.nc')
significant_w_seasons_kd = seasonal_kd['pvalue'] <= 0.05  # 95% confidence
trend_sig_seasons_kd = seasonal_kd['slope'].where(significant_w_seasons_kd)

In [55]:
print("Longitude range:", float(stat_trend_kd['longitude'].min()), "to", float(stat_trend_kd['longitude'].max()))
print("Latitude range :", float(stat_trend_kd['latitude'].min()), "to", float(stat_trend_kd['latitude'].max()))

Longitude range: -14.994791030883789 to -2.005207061767578
Latitude range : 48.00520706176758 to 58.99479293823242


In [56]:
min_slope = float(stat_trend_kd['slope'].min(skipna=True))
max_slope = float(stat_trend_kd['slope'].max(skipna=True))

print(f"True slope boundaries: Min = {min_slope:.5f} | Max = {max_slope:.5f}")

# Highly Recommended: Check the 2nd and 98th percentiles
# This prevents extreme single-pixel outliers from forcing you to choose too wide of a colorbar range
p2, p98 = np.nanpercentile(stat_trend_kd['slope'].values, [2, 98])
print(f"96% of data falls between: {p2:.5f} and {p98:.5f}")

True slope boundaries: Min = -0.01475 | Max = 0.01502
96% of data falls between: -0.00120 and 0.00079


In [57]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = stat_trend_kd['slope'].plot(
    ax=ax,
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    add_colorbar=False,
    zorder=2
)
## add landmass
landmass.plot(ax=ax,
        facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
    )
## set x and y marks
ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

    # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True  # Force visibility toggles on
    )
## fix spine color and width
for spine in ax.spines.values():
    spine.set_edgecolor('#d3d3d3')
    spine.set_linewidth(0.8)
## fix colorbar
cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ K$_{d490}$ ($m^{-1}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'K$_{d490}$ Linear Trend Map', fontweight='bold', fontsize=12, pad=15)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)


plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Weekly_Trend_KD490.png', dpi=1200, bbox_inches='tight')
plt.show()

In [58]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = trend_sig_kd.plot(
    ax=ax,
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    #cbar_kwargs={'label': r'$\Delta$ K$_{d490}$ ($m^{-1}$ $y^{-1}$)'},
    add_colorbar=False,
    zorder=2
)

landmass.plot(ax=ax,
       facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
   )

## fix spine color and width
for spine in ax.spines.values():
   spine.set_edgecolor('#d3d3d3')
   spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

    # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True  # Force visibility toggles on
    )

## fix colorbar


cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ K$_{d490}$ ($m^{-1}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)


ax.set_title(r'K$_{d490}$ Linear Trend Map: p<=0.05', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Weekly_Trend_KD490.png', dpi=1200, bbox_inches='tight')
plt.show()

In [59]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

#need to do this so lat and long are only on the outside
axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    ## determine what column and row it is to assign lables
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

    ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)
    ## add satellite data
    seasonal_kd['slope'].sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.002, vmax=0.002, add_colorbar=False, zorder=2
    )
    ## add season title per plot
    ax.set_title(f'{season.capitalize()}', fontsize=11)

## Axis formating!
    ##only put lat on the far left column, where col ==0
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
    ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')

    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

    # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True  # Force visibility toggles on
    )

    ## make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)

   # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])

cbar = fig.colorbar(
    plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.002, vmax=0.002)),
    cax=cbar_ax,
    orientation='vertical',
    extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ K$_{d490}$ ($m^{-1}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'K$_{d490}$ Seasonal Linear Trend Maps', fontweight='bold', fontsize=12, x=0.464, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Seasonal_Trend_KD490.png',
            dpi=1200, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [60]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(7, 8))

fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

#need to do this so lat and long are only on the outside
axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    trend_sig_seasons_kd.sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.002, vmax=0.002, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)
    if col == 0:
       ax.set_ylabel(r'Latitude', fontsize=10)
    else:
       ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
       ax.set_xlabel(r'Longitude', fontsize=10)
    else:
       ax.set_xlabel('')


    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))


   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
    )


   ## make the box around map light grey
    for spine in ax.spines.values():
       spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
       spine.set_linewidth(0.8)


  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])

cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.002, vmax=0.002)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ K$_{d490}$ ($m^{-1}$ $y^{-1}$)', fontsize=10)


fig.suptitle(r'K$_{d490}$ Seasonal Linear Trend Maps: p<=0.05', fontweight='bold', fontsize=12, x=0.456, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Seasonal_Trend_KD490.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)

plt.show()

### Chlorophyll-a Plots

In [61]:
chla_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/chlorophyll/*.nc'))
chla_ds = xr.open_mfdataset(chla_file_list, concat_dim='time', combine='nested')

weekly_chla = chla_ds['CHL'].resample(time='1W').mean()
monthly_chla = chla_ds['CHL'].resample(time='1ME').mean()
seasonal_chla = chla_ds['CHL'].resample(time='QS-DEC').mean()
seasonal_chla['season'] = seasonal_chla['time'].dt.season

weekly_trend_chla = xr.open_dataset('thesis_data/Chl_a_trend_with_stats.nc')

w_significant_chla = weekly_trend_chla['pvalue'] <= 0.05  # 95% confidence
w_trend_sig_chla = weekly_trend_chla['slope'].where(w_significant_chla)

seasonal_trend_chla = xr.open_dataset('thesis_data/Chl_a_seasonal_trends.nc')
sea_significant_chla = seasonal_trend_chla['pvalue'] <= 0.05  # 95% confidence
sea_trend_sig_chla = seasonal_trend_chla['slope'].where(sea_significant_chla)

In [62]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = weekly_trend_chla['slope'].plot(
    ax=ax,
    cmap='RdBu_r',
    center=0,
    vmin=-0.04, vmax=0.04,
    #cbar_kwargs={'label': r'$\Delta$ Chl-$\mathregular{\mathit{a}}$ (mg $m^{-3}$ $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)

landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar

cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ Chl-$\mathregular{\mathit{a}}$ (mg $m^{-3}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'Chl-$\mathregular{\mathit{a}}$ Linear Trend Map', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Weekly_Trend_Chla.png', dpi=1200, bbox_inches='tight')
plt.show()

In [63]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = w_trend_sig_chla.plot(
    ax=ax,
    cmap='RdBu_r',
    center=0,
    vmin=-0.04, vmax=0.04,
    #cbar_kwargs={'label': r'$\Delta$ Chl-$\mathregular{\mathit{a}}$ (mg $m^{-3}$ $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)

landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )


## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)


ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))


   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )
## fix colorbar
cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label( r'$\Delta$ Chl-$\mathregular{\mathit{a}}$ (mg $m^{-3}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'Chl-$\mathregular{\mathit{a}}$ Linear Trend Map: p<=0.05', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Weekly_Trend_Chla.png', dpi=1200, bbox_inches='tight')
plt.show()

In [64]:
seasons = ['autumn', 'winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

axes_grid = axes.reshape(2,2)
for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    seasonal_trend_chla['slope'].sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.04, vmax=0.04, add_colorbar=False, zorder=2)

    ax.set_title(f'{season.capitalize()}', fontsize=11)
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')


    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))


   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True)  # Force visibility toggles on)

   ## make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)

  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])

cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.04, vmax=0.04)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label(r'$\Delta$ Chl-$\mathregular{\mathit{a}}$ (mg $m^{-3}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'Chl-$\mathregular{\mathit{a}}$ Seasonal Linear Trend Maps', fontweight='bold', fontsize=12, x=0.464, y=.98)

fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Seasonal_Trend_Chla.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [65]:
seasons = ['autumn', 'winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

axes_grid = axes.reshape(2,2)
for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    sea_trend_sig_chla.sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.04, vmax=0.04, add_colorbar=False, zorder=2)

    ax.set_title(f'{season.capitalize()}', fontsize=11)
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')


    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))


   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True)  # Force visibility toggles on)

   ## make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)

  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])

cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.04, vmax=0.04)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label(r'$\Delta$ Chl-$\mathregular{\mathit{a}}$ (mg $m^{-3}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'Chl-$\mathregular{\mathit{a}}$ Seasonal Linear Trend Maps: p<=0.05', fontweight='bold', fontsize=12, x=0.464, y=.98)

fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Seasonal_Trend_Chla.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)
plt.show()

### CDOM Plots

In [66]:
cdom_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/CDOM/*.nc'))
cdom_ds = xr.open_mfdataset(cdom_file_list, concat_dim='time', combine='nested')

weekly_cdom = cdom_ds['CDM'].resample(time='1W').mean(skipna=True)
monthly_cdom = cdom_ds['CDM'].resample(time='1ME').mean(skipna=True)
seasonal_cdom = cdom_ds['CDM'].resample(time='QS-DEC').mean(skipna=True)
seasonal_cdom['season'] = seasonal_cdom['time'].dt.season

weekly_trend_cdom = xr.open_dataset('thesis_data/CDOM_trend_with_stats.nc')

w_significant_cdom = weekly_trend_cdom['pvalue'] <= 0.05  # 95% confidence
w_trend_sig_cdom = weekly_trend_cdom['slope'].where(w_significant_cdom)

seasonal_trend_cdom = xr.open_dataset('thesis_data/CDOM_seasonal_trends.nc')
sea_significant_cdom = seasonal_trend_cdom['pvalue'] <= 0.05  # 95% confidence
sea_trend_sig_cdom = seasonal_trend_cdom['slope'].where(sea_significant_cdom)

In [67]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)


im = weekly_trend_cdom['slope'].plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.003, vmax=0.003,
    #cbar_kwargs={'label': r'$\Delta$ CDOM ($m^{-1}$ $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)
landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar
cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ CDOM ($m^{-1}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'CDOM Linear Trend Map', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Weekly_Trend_CDOM.png', dpi=1200, bbox_inches='tight')
plt.show()

In [68]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)


im = w_trend_sig_cdom.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.003, vmax=0.003,
    #cbar_kwargs={'label': r'$\Delta$ CDOM ($m^{-1}$ $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)
landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar
cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ CDOM ($m^{-1}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'CDOM Linear Trend Map: p<=0.05', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Weekly_Trend_CDOM.png', dpi=1200, bbox_inches='tight')
plt.show()

In [69]:
seasons = ['autumn', 'winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]


   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    seasonal_trend_cdom['slope'].sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.003, vmax=0.003, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)
## Axis formating!
   ##only put lat on the far left column, where col ==0
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')

    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

   ## make the box around map light grey
    for spine in ax.spines.values():
       spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
       spine.set_linewidth(0.8)

  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])


cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.003, vmax=0.003)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ CDOM ($m^{-1}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'CDOM Seasonal Linear Trend Maps', fontweight='bold', fontsize=12, x=0.464, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Seasonal_Trend_CDOM.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)

plt.show()

In [70]:
seasons = ['autumn', 'winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]


   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    sea_trend_sig_cdom.sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.003, vmax=0.003, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)
## Axis formating!
   ##only put lat on the far left column, where col ==0
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')

    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

   ## make the box around map light grey
    for spine in ax.spines.values():
       spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
       spine.set_linewidth(0.8)

  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])


cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.003, vmax=0.003)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ CDOM ($m^{-1}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'CDOM Seasonal Linear Trend Maps: p<=0.05', fontweight='bold', fontsize=12, x=0.464, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Seasonal_Trend_CDOM.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)

plt.show()

### SPM Plots

In [71]:
spm_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/SPM/*.nc'))
spm_ds = xr.open_mfdataset(spm_file_list, concat_dim='time', combine='nested')

weekly_spm = spm_ds['SPM'].resample(time='1W').mean(skipna=True)
monthly_spm = spm_ds['SPM'].resample(time='1ME').mean(skipna=True)
seasonal_spm = spm_ds['SPM'].resample(time='QS-DEC').mean(skipna=True)
seasonal_spm['season'] = seasonal_spm['time'].dt.season

weekly_trend_spm = xr.open_dataset('thesis_data/SPM_trend_with_stats.nc')

w_significant_spm = weekly_trend_spm['pvalue'] <= 0.05  # 95% confidence
w_trend_sig_spm = weekly_trend_spm['slope'].where(w_significant_spm)

seasonal_trend_spm = xr.open_dataset('thesis_data/SPM_seasonal_trends.nc')
sea_significant_spm = seasonal_trend_spm['pvalue'] <= 0.05  # 95% confidence
sea_trend_sig_spm = seasonal_trend_spm['slope'].where(sea_significant_spm)

In [72]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = weekly_trend_spm['slope'].plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.08, vmax=0.08,
    #cbar_kwargs={'label': r'$\Delta$ SPM (g $m^{-3}$ $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)
landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar

cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ SPM (g $m^{-3}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'SPM Linear Trend Map', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Weekly_Trend_SPM.png', dpi=1200, bbox_inches='tight')
plt.show()

In [73]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = w_trend_sig_spm.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.08, vmax=0.08,
    #cbar_kwargs={'label': r'$\Delta$ SPM (g $m^{-3}$ $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)
landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar

cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ SPM (g $m^{-3}$ $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)

ax.set_title(r'SPM Linear Trend Map: p<=0.05', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Weekly_Trend_SPM.png', dpi=1200, bbox_inches='tight')
plt.show()

In [74]:
seasons = ['autumn', 'winter', 'spring', 'summer']

fig, axes = plt.subplots(2, 2, figsize=(7, 8) )

fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])

panel_labels = ['(a)', '(b)', '(c)', '(d)']
#need to do this so lat and long are only on the outside
axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    seasonal_trend_spm['slope'].sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.08, vmax=0.08, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)

## Axis formating!
   ##only put lat on the far left column, where col ==0
    if col == 0:
       ax.set_ylabel(r'Latitude', fontsize=10)
    else:
       ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
       ax.set_xlabel(r'Longitude', fontsize=10)
    else:
       ax.set_xlabel('')

    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True ) # Force visibility toggles on)

#make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)


  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')


# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])


cbar = fig.colorbar(plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.08, vmax=0.08)),
                    cax=cbar_ax,
                    orientation='vertical',
                    extend='both')

cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ SPM (g $m^{-3}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'SPM Seasonal Linear Trend Maps', fontweight='bold', fontsize=12, x=0.464, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Seasonal_Trend_SPM.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)

plt.show()

In [75]:
seasons = ['autumn', 'winter', 'spring', 'summer']

fig, axes = plt.subplots(2, 2, figsize=(7, 8) )

fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])

panel_labels = ['(a)', '(b)', '(c)', '(d)']
#need to do this so lat and long are only on the outside
axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    sea_trend_sig_spm.sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.08, vmax=0.08, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)

## Axis formating!
   ##only put lat on the far left column, where col ==0
    if col == 0:
       ax.set_ylabel(r'Latitude', fontsize=10)
    else:
       ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
       ax.set_xlabel(r'Longitude', fontsize=10)
    else:
       ax.set_xlabel('')

    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True ) # Force visibility toggles on)

#make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)


  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')


# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])


cbar = fig.colorbar(plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.08, vmax=0.08)),
                    cax=cbar_ax,
                    orientation='vertical',
                    extend='both')

cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ SPM (g $m^{-3}$ $y^{-1}$)', fontsize=10)

fig.suptitle(r'SPM Seasonal Linear Trend Maps: p<=0.05', fontweight='bold', fontsize=12, x=0.456, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Seasonal_Trend_SPM.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)

plt.show()

### SST Plots

In [76]:
sst_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/SST2/*.nc'))
sst_ds = xr.open_mfdataset(sst_file_list, concat_dim='time', combine='nested')

## Weekly, Monthly, Seasonal
weekly_sst = sst_ds['analysed_sst'].resample(time='1W').mean()
monthly_sst = sst_ds['analysed_sst'].resample(time='1ME').mean()
seasonal_sst = sst_ds['analysed_sst'].resample(time='QS-DEC').mean()
seasonal_sst['season'] = seasonal_sst['time'].dt.season

weekly_trend_sst = xr.open_dataset('thesis_data/SST_trend_with_stats.nc')
w_significant_sst = weekly_trend_sst['pvalue'] < 0.05  # 95% confidence
w_trend_sig_sst = weekly_trend_sst['slope'].where(w_significant_sst)

#### Never did the seasonal trends for SST, going to run and save now


def fit_with_uncertainty(y, x, min_points=20):
    mask = np.isfinite(y)
    if mask.sum() < min_points:
        # slope, intercept, r, p, stderr_slope, stderr_intercept
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

    res = stats.linregress(x[mask], y[mask])
    return (
        res.slope,
        res.intercept,
        res.rvalue,
        res.pvalue,
        res.stderr,             # standard error of the slope
        getattr(res, "intercept_stderr", np.nan)  # available in SciPy >=1.9
    )


def assign_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'


season_labels = seasonal_sst['time'].dt.month.to_series().apply(assign_season)
seasonal_sst = seasonal_sst.assign_coords(season=('time', season_labels.values))

seasonal_trends = {}

for season in ['winter', 'spring', 'summer', 'autumn']:
    season_data = seasonal_sst.where(seasonal_sst['season'] == season, drop=True)

    # Compute decimal years for regression
    t = season_data['time']
    time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / \
                 (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(
                     t.dt.year.astype(str) + '-01-01'))
    x_season = time_years.values.astype(float)

    # Apply linear regression with uncertainty
    slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
        fit_with_uncertainty,
        season_data,
        input_core_dims=[['time']],
        output_core_dims=[[], [], [], [], [], []],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float, float, float, float, float, float],
        kwargs={'x': x_season}
    )

    # Store per-season results
    seasonal_trends[season] = xr.Dataset({
        'slope': slope,
        'intercept': intercept,
        'rvalue': rvalue,
        'pvalue': pvalue,
        'slope_stderr': slope_stderr,
        'intercept_stderr': intercept_stderr
    })

season_ds_list = []
for season, ds in seasonal_trends.items():
    ds = ds.expand_dims({'season': [season]})
    season_ds_list.append(ds)

seasonal_trend_sst = xr.concat(season_ds_list, dim='season')

seasonal_trend_sst = seasonal_trend_sst.compute()

encoding = {v: {"zlib": True, "complevel": 4} for v in seasonal_trend_sst.data_vars}
seasonal_trend_sst.to_netcdf('SST_seasonal_trends.nc', encoding=encoding)

#### Back to loading data

In [77]:
seasonal_trend_sst = xr.open_dataset('SST_seasonal_trends.nc')


sea_significant_sst = seasonal_trend_sst['pvalue'] < 0.05  # 95% confidence
sea_trend_sig_sst = seasonal_trend_sst['slope'].where(sea_significant_sst)

In [78]:
weekly_sst = weekly_sst - 273.15
## transforming it to celsius
weekly_sst.attrs['units'] = 'Celsius'
weekly_sst.attrs['long_name'] = 'Analyzed Sea Surface Temperature'

In [79]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = weekly_trend_sst['slope'].plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.04, vmax=0.04,
    #cbar_kwargs={'label': r'$\Delta$ SST ($^\circ$K $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)

landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar
cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ SST ($^\circ$C $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)


ax.set_title(r'SST Linear Trend Map', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)

plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Weekly_Trend_SST.png', dpi=1200, bbox_inches='tight')
plt.show()

In [80]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)

im = w_trend_sig_sst.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.04, vmax=0.04,
    #cbar_kwargs={'label': r'$\Delta$ SST ($^\circ$K $y^{-1}$)'}
    add_colorbar=False,
    zorder=2
)

landmass.plot(ax=ax,
      facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1
  )

## fix spine color and width
for spine in ax.spines.values():
  spine.set_edgecolor('#d3d3d3')
  spine.set_linewidth(0.8)

ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))

   # direction='out' pushes them away from the data; length and width make them crisp
ax.tick_params(
       axis='both',
       which='major',
       direction='out',
       length=3,
       width=0.8,
       colors='#d3d3d3',
       labelcolor='#333333',
       labelsize=9,
       bottom=True, left=True  # Force visibility toggles on
   )

## fix colorbar
cbar_ax = fig.add_axes([0.76, 0.12, 0.025, 0.73])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label(r'$\Delta$ SST ($^\circ$C $y^{-1}$)', fontsize=10)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)


ax.set_title(r'SST Linear Trend Map: p<=0.05', fontweight='bold', fontsize=12)

ax.set_ylabel(r'Latitude', fontsize=10)
ax.set_xlabel(r'Longitude', fontsize=10)

plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Weekly_Trend_SST.png', dpi=1200, bbox_inches='tight')
plt.show()

In [81]:

seasons = ['autumn', 'winter', 'spring', 'summer']
## edit to add plot labels
panel_labels = ['(a)', '(b)', '(c)', '(d)']

fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

#need to do this so lat and long are only on the outside
axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    seasonal_trend_sst['slope'].sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.04, vmax=0.04, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)

## Axis formating!
   ##only put lat on the far left column, where col ==0
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')


    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))


   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True  # Force visibility toggles on
    )


   ## make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)


  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])

cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.04, vmax=0.04)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ SST ($^\circ$C $y^{-1}$)', fontsize=10)

fig.suptitle(r'SST Seasonal Linear Trend Maps', fontweight='bold', fontsize=12, x=0.464, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/Seasonal_Trend_SST.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [82]:

seasons = ['autumn', 'winter', 'spring', 'summer']
## edit to add plot labels
panel_labels = ['(a)', '(b)', '(c)', '(d)']

fig, axes = plt.subplots(2, 2, figsize=(7, 8) )
fig.set_layout_engine('constrained', rect=[0,0,0.88,0.95])
panel_labels = ['(a)', '(b)', '(c)', '(d)']

#need to do this so lat and long are only on the outside
axes_grid = axes.reshape(2,2)

for i, season, label in zip(range(4), seasons, panel_labels):
    row = i // 2
    col = i % 2
    ax = axes_grid[row, col]

   ## add landmasses
    landmass.plot(ax=ax, facecolor='#EAEAF2', edgecolor='#808080', linewidth=0.25, zorder=1)

    sea_trend_sig_sst.sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.04, vmax=0.04, add_colorbar=False, zorder=2
    )
    ax.set_title(f'{season.capitalize()}', fontsize=11)

## Axis formating!
   ##only put lat on the far left column, where col ==0
    if col == 0:
        ax.set_ylabel(r'Latitude', fontsize=10)
    else:
        ax.set_ylabel("")
   ## only put long on the very bottom, where row ==1
    if row ==1:
        ax.set_xlabel(r'Longitude', fontsize=10)
    else:
        ax.set_xlabel('')


    ax.xaxis.set_major_locator(FixedLocator([-15, -13, -11, -9, -7, -5, -3, -1]))
    ax.yaxis.set_major_locator(FixedLocator([48, 50, 52, 54, 56, 58]))


   # direction='out' pushes them away from the data; length and width make them crisp
    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=0.8,
        colors='#d3d3d3',
        labelcolor='#333333',
        labelsize=9,
        bottom=True, left=True  # Force visibility toggles on
    )


   ## make the box around map light grey
    for spine in ax.spines.values():
        spine.set_edgecolor('#d3d3d3')  # Make it a thin, light gray
        spine.set_linewidth(0.8)


  # add panel label text
    ax.text(1.02, 0.95, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='left')

# shared colorbar
cbar_ax = fig.add_axes([0.87, 0.11, 0.025, 0.78])

cbar = fig.colorbar(
   plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.04, vmax=0.04)),
   cax=cbar_ax,
   orientation='vertical',
   extend='both',
)
cbar.ax.spines['outline'].set_edgecolor('#d3d3d3')
cbar.ax.spines['outline'].set_linewidth(0.6)
cbar.set_label( r'$\Delta$ SST ($^\circ$C $y^{-1}$)', fontsize=10)

fig.suptitle(r'SST Seasonal Linear Trend Maps: p<=0.05', fontweight='bold', fontsize=12, x=0.464, y=0.98)
fig.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/StatisticallySignificant_Seasonal_Trend_SST.png', dpi=1200, bbox_inches='tight', pad_inches=0.2)
plt.show()

C:\Users\25298423\AppData\Local\Temp\ipykernel_13456\2064894946.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(2, 2, figsize=(7, 8) )


## Part 2: Temporal Trajectories and Breakpoints

In [5]:
import gc
import sys

# Force Python's garbage collector to release dead memory links instantly
gc.collect()

195

In [6]:
# Open the dataset with chunks to engage Dask lazily
sat_ds = xr.open_dataset('satellite_weekly_means.nc')

In [7]:
print(sat_ds.rio.crs)

None


In [8]:
sat_ds = sat_ds.rio.set_spatial_dims(x_dim='x', y_dim='y')
sat_ds.rio.write_crs("EPSG:3035", inplace=True)
print("Current CRS:", sat_ds.rio.crs)

Current CRS: EPSG:3035


In [9]:
## now island boundary

In [10]:
island = gpd.read_file('thesis_data/island_boundary.gpkg')

In [11]:
island.to_crs("EPSG:3035", inplace=True)

In [12]:
coastal_buffer = island.buffer(5000)

In [13]:


fig, ax = plt.subplots(figsize=(8, 10))

coastal_buffer.plot(ax=ax, color="lightgray", edgecolor="black")
island.plot(ax=ax, color='gray', edgecolor= 'black')

plt.show()

In [14]:
import rasterio.features
import xarray as xr

# 1. Generate the 2D numpy boolean mask using rasterio
coastal_mask_np = rasterio.features.rasterize(
    coastal_buffer.geometry,
    out_shape=(sat_ds.rio.height, sat_ds.rio.width),
    transform=sat_ds.rio.transform(),
    fill=0,            # Pixels outside the buffer get 0 (False)
    default_value=1,   # Pixels inside the buffer get 1 (True)
    all_touched=True
).astype(bool)

# 2. Re-wrap it into an Xarray DataArray using your dataset's exact spatial coordinates
coastal_mask = xr.DataArray(
    coastal_mask_np,
    coords={'y': sat_ds.y, 'x': sat_ds.x},
    dims=['y', 'x']
)

# 3. Sanity check: verify the shape matches your satellite dataset
print("Mask shape:", coastal_mask.shape)
print("Satellite shape:", sat_ds.rio.height, sat_ds.rio.width)

Mask shape: (1535, 1247)
Satellite shape: 1535 1247


In [15]:
## chunk it!
sat_ds = sat_ds.chunk({'time': 20, 'y': -1, 'x': -1})
coastal_mask = coastal_mask.chunk({'y': -1, 'x': -1})

In [16]:

# make mask binary weights
coastal_weight = coastal_mask.astype(int)

In [17]:


## need to slice off the irish sea pixels,
open_ocean_bool = (coastal_mask == False) & (sat_ds.x <= 3200000)
open_ocean_weight = open_ocean_bool.astype(int)

print("Extracting full Irish coastal weekly averages...")
coastal_ts_ds = sat_ds.weighted(coastal_weight).mean(dim=['y', 'x'], skipna=True).compute(scheduler='sync')

print("Extracting clean Atlantic/Celtic Open Ocean weekly averages...")
ocean_ts_ds = sat_ds.weighted(open_ocean_weight).mean(dim=['y', 'x'], skipna=True).compute(scheduler='sync')

print("Extraction successful!")

Extracting full Irish coastal weekly averages...
Extracting clean Atlantic/Celtic Open Ocean weekly averages...
Extraction successful!


In [18]:
coastal_ts_ds

<xarray.Dataset> Size: 70kB
Dimensions:      (time: 1462)
Coordinates:
  * time         (time) datetime64[ns] 12kB 1997-09-07 1997-09-14 ... 2025-09-07
    spatial_ref  int64 8B 0
Data variables:
    kd490        (time) float64 12kB nan nan 0.136 ... 0.09601 0.1263 0.1731
    chl_a        (time) float64 12kB nan nan 1.975 1.973 ... 1.212 2.16 3.27
    spm          (time) float64 12kB nan nan 4.031 5.837 ... 1.131 2.391 1.892
    sst          (time) float64 12kB 288.4 288.3 288.1 ... 289.4 289.1 288.8
    cdom         (time) float64 12kB nan nan 0.1573 ... 0.06619 0.1305 0.127

In [19]:
# 1. Define your target oceanographic attributes
attributes = ['kd490', 'chl_a', 'spm', 'sst', 'cdom']

# 2. Convert the full 1D Xarray dataset to a multi-column Pandas DataFrame
# (Assuming your xarray variables match these names lowercase/uppercase)
df_raw = coastal_ts_ds[attributes].to_dataframe()

# Ensure strict calendar continuity across the entire 28-year timeline
# --- FIX: Create a continuous 7-day grid anchored to your true data start date ---
# This prevents pandas from shifting your natural dates to arbitrary calendar Sundays
start_date = df_raw.index.min()
end_date = df_raw.index.max()
continuous_dates = pd.date_range(start=start_date, end=end_date, freq='7D')

# Reindex using the true continuous 7-day environmental grid
df_continuous = df_raw.reindex(continuous_dates)

## SLM take 1

**kd490 and chla are measured using the same blue to green band ratios most of the time, so having such a high correlation is partly an artifact of the algorithms at play. This is why in-situ validation is important**

## Start Here for SLMs - includes atmo variables

In [20]:
## what about adding precipitation to the mix?

atmo = gpd.read_parquet("C:/Users/25298423/PycharmProjects/JupyterProject1/atmo_stage3_projected.parquet")
atmo

,station,date,rain,mean_wind_speed_kt,high_ten_min_mean_kt,direction_highten_deg,highest_gust,latitude,longitude,rain_7d,geometry,x_3035,y_3035,wind_u,wind_v
0,ROCHES POINT,1955-12-01,9.1,11.5,18.0,170.0,24.0,51.793,-8.244,NaN,POINT (3077434.62 3343802.619),3.077435e+06,3.343803e+06,-3.125667,1.772654e+01
1,ROCHES POINT,1955-12-02,0.9,12.1,23.0,210.0,32.0,51.793,-8.244,NaN,POINT (3077434.62 3343802.619),3.077435e+06,3.343803e+06,11.500000,1.991858e+01
2,ROCHES POINT,1955-12-03,0.0,8.0,16.0,270.0,21.0,51.793,-8.244,NaN,POINT (3077434.62 3343802.619),3.077435e+06,3.343803e+06,16.000000,2.939152e-15
3,ROCHES POINT,1955-12-04,1.5,6.3,16.0,210.0,23.0,51.793,-8.244,NaN,POINT (3077434.62 3343802.619),3.077435e+06,3.343803e+06,8.000000,1.385641e+01
4,ROCHES POINT,1955-12-05,0.8,16.5,23.0,220.0,32.0,51.793,-8.244,NaN,POINT (3077434.62 3343802.619),3.077435e+06,3.343803e+06,14.784115,1.761902e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167941,DUBLIN AIRPORT,2026-05-27,0.0,8.9,15.0,70.0,21.0,53.428,-6.241,1.7,POINT (3251781.854 3489405.462),3.251782e+06,3.489405e+06,-14.095389,-5.130302e+00
167942,DUBLIN AIRPORT,2026-05-28,0.0,9.3,17.0,230.0,26.0,53.428,-6.241,0.0,POINT (3251781.854 3489405.462),3.251782e+06,3.489405e+06,13.022756,1.092739e+01
167943,DUBLIN AIRPORT,2026-05-29,0.0,9.6,15.0,260.0,27.0,53.428,-6.241,0.0,POINT (3251781.854 3489405.462),3.251782e+06,3.489405e+06,14.772116,2.604723e+00
167944,DUBLIN AIRPORT,2026-05-30,1.7,6.1,13.0,220.0,21.0,53.428,-6.241,1.7,POINT (3251781.854 3489405.462),3.251782e+06,3.489405e+06,8.356239,9.958578e+00


atmo['sumrain_7d'] = atmo.groupby('station')['rain'].transform(lambda x: x.rolling(7).sum())
atmo['meanwind_7d'] = atmo.groupby('station')['mean_wind_speed_kt'].transform(lambda x: x.rolling(7).mean())
atmo['maxburst_7d'] = atmo.groupby('station')['high_ten_min_mean_kt'].transform(lambda x: x.rolling(7).max())
atmo['mean_wind_u_7d'] = atmo.groupby('station')['wind_u'].transform(lambda x: x.rolling(7).mean())
atmo['mean_wind_v_7d'] = atmo.groupby('station')['wind_v'].transform(lambda x: x.rolling(7).mean())

In [21]:
atmo['date']=pd.to_datetime(atmo['date'])

In [22]:
rain = atmo[['station','date','rain', 'mean_wind_speed_kt', 'highest_gust','latitude','longitude']]

In [23]:
rain.set_index("date", inplace=True)


In [24]:
rain.index = pd.to_datetime(rain.index)


In [25]:
rain_daily_mean = rain.groupby(rain.index).mean(numeric_only=True)


In [26]:
df_rolling = pd.DataFrame(index=rain_daily_mean.index)


In [27]:
df_rolling['sumrain_7d'] = rain_daily_mean['rain'].rolling(window=7, min_periods=7).sum()
df_rolling['meanwind_7d'] = rain_daily_mean['mean_wind_speed_kt'].rolling(window=7, min_periods=7).mean()
df_rolling['maxburst_7d'] = rain_daily_mean['highest_gust'].rolling(window=7, min_periods=7).max()

In [28]:
df_weather_aligned = pd.DataFrame(index=df_continuous.index)


In [29]:
df_weather_aligned['sumrain_7d'] = df_weather_aligned.index.map(
    lambda sat_date: df_rolling.at[sat_date, 'sumrain_7d']
    if sat_date in df_rolling.index else None
)

# 3. Map Column 2: 1-Week Hydrological Lag (Previous-week weather)
# Example: Sat date 1997-09-14 pulls the daily rolling rain from 1997-09-07
df_weather_aligned['lag_sumrain_7d'] = df_weather_aligned.index.map(
    lambda sat_date: df_rolling.at[sat_date - pd.Timedelta(days=7), 'sumrain_7d']
    if (sat_date - pd.Timedelta(days=7)) in df_rolling.index else None
)

df_weather_aligned['meanwind_7d'] = df_weather_aligned.index.map(
    lambda sat_date: df_rolling.at[sat_date, 'meanwind_7d']
    if sat_date in df_rolling.index else None
)

df_weather_aligned['maxburst_7d'] = df_weather_aligned.index.map(
    lambda sat_date: df_rolling.at[sat_date, 'maxburst_7d']
    if sat_date in df_rolling.index else None
)
# 4. Append both columns directly into your continuous master dataframe
df_continuous['sumrain_7d'] = df_weather_aligned['sumrain_7d']
df_continuous['lag_sumrain_7d'] = df_weather_aligned['lag_sumrain_7d']
df_continuous['meanwind_7d'] = df_weather_aligned['meanwind_7d']
df_continuous['maxburst_7d'] = df_weather_aligned['maxburst_7d']

In [30]:
df_continuous['sst_C'] = df_continuous['sst'] - 273.15


## SLM take 3 - ARIMA forcasting added

In [31]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
attributes = ['cdom', 'chl_a', 'kd490', 'spm', 'sst_C', 'sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']
# Ensure DataFrame Index is a clean DatetimeIndex
df_continuous.index = pd.to_datetime(df_continuous.index)
original_start_date = df_continuous.index.min()
original_end_date = df_continuous.index.max()

# Define padding length: 104 weeks (~2 years on both sides)
pad_weeks = 104

trends = {}
seasonals = {}
residuals = {}

print("Beginning Dual-Ended Predictive STL Decomposition...")

# Automatically extract your exact dataset cadence (e.g., '7D')
native_freq = df_continuous.index.freq
if native_freq is None:
    native_freq = pd.infer_freq(df_continuous.index)
    if native_freq is None:
        # Fallback to your explicit generation step if inference struggles
        native_freq = '7D'

for attr in attributes:
    print(f" -> Decomposing {attr}...")

    # 1. Clean the raw data slice
    raw_series = df_continuous[attr].interpolate(method='time').bfill().ffill()

    # Marry the inferred/extracted explicit frequency to the series index
    raw_series.index.freq = native_freq

    # =================================================================
    # A. FORECASTING FOR THE TRAILING EDGE (End of Data)
    # =================================================================
    try:
        model_fwd = sm.tsa.statespace.SARIMAX(
            raw_series, order=(1,0,0), seasonal_order=(0,1,0,52),
            enforce_stationarity=False, enforce_invertibility=False
        )
        fit_fwd = model_fwd.fit(disp=False)
        forecast_raw = fit_fwd.forecast(steps=pad_weeks)

        # Build future dates building cleanly off your exact end timestamp
        future_dates = pd.date_range(
            start=original_end_date + pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        forecast_extended = pd.Series(forecast_raw.values, index=future_dates)
    except:
        # Robust structural fallback matched to your exact step frequency
        future_dates = pd.date_range(
            start=original_end_date + pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        forecast_extended = pd.Series(np.tile(raw_series.iloc[-52:].values, 2), index=future_dates)

    # =================================================================
    # B. BACKCASTING FOR THE LEADING EDGE (Start of Data)
    # =================================================================
    try:
        # Drop the DatetimeIndex and convert to raw values before reversing to avoid warnings
        reversed_values = raw_series.values[::-1]

        model_bwd = sm.tsa.statespace.SARIMAX(
            reversed_values, order=(1,0,0), seasonal_order=(0,1,0,52),
            enforce_stationarity=False, enforce_invertibility=False
        )
        fit_bwd = model_bwd.fit(disp=False)
        backcast_raw = fit_bwd.forecast(steps=pad_weeks)

        # Build historical dates stepping cleanly backward from your start timestamp
        backcast_dates = pd.date_range(
            end=original_start_date - pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        backcast_extended = pd.Series(backcast_raw.values[::-1], index=backcast_dates)
    except:
        # Robust structural fallback matched to your exact step frequency
        backcast_dates = pd.date_range(
            end=original_start_date - pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        backcast_extended = pd.Series(np.tile(raw_series.iloc[:52].values, 2), index=backcast_dates)

    # =================================================================
    # C. STITCH AND DECOMPOSE
    # =================================================================
    # Merge everything into a single timeline and ensure a consistent step pattern
    extended_series = pd.concat([backcast_extended, raw_series, forecast_extended]).sort_index()
    extended_series.index.freq = native_freq

    # Fit the Robust STL on the stabilized extended dataframe
    res_extended = STL(extended_series, period=52, trend=105, robust=True).fit()

    # Crop out both padding ends to restore the exact original 28-year dimensions
    trends[attr] = res_extended.trend.loc[original_start_date:original_end_date]
    seasonals[attr] = res_extended.seasonal.loc[original_start_date:original_end_date]
    residuals[attr] = res_extended.resid.loc[original_start_date:original_end_date]

# =====================================================================
# 2. CONSOLIDATE INTO MULTI-INDEX STRUCTURING
# =====================================================================
df_trends = pd.DataFrame(trends, index=df_continuous.index)
df_seasonals = pd.DataFrame(seasonals, index=df_continuous.index)
df_residuals = pd.DataFrame(residuals, index=df_continuous.index)

df_stl_wrain = pd.concat(
    {'trend': df_trends, 'seasonal': df_seasonals, 'residual': df_residuals},
    axis=1
).swaplevel(0, 1, axis=1).sort_index(axis=1)

print("Dual-ended stabilization complete! Calculating matrices...")
trend_correlation = df_stl_wrain.xs('trend', level=1, axis=1).corr()
trend_correlation.to_csv('satellite_trend_correlation.csv', index=False)

Beginning Dual-Ended Predictive STL Decomposition...
 -> Decomposing cdom...
 -> Decomposing chl_a...
 -> Decomposing kd490...


C:\Users\25298423\PycharmProjects\JupyterProject1\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


 -> Decomposing spm...
 -> Decomposing sst_C...
 -> Decomposing sumrain_7d...
 -> Decomposing lag_sumrain_7d...
 -> Decomposing meanwind_7d...
 -> Decomposing maxburst_7d...
Dual-ended stabilization complete! Calculating matrices...


In [32]:
## raw weekly values


# Define your exact target oceanographic attributes in order
attributes = ['cdom', 'chl_a', 'kd490', 'spm', 'sst_C', 'sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']

# 1. Reconstruct the clean, cloud-filled continuous data from your STL components
# (This ensures we use the exact data that went into your trends, minus data gaps)
df_clean_raw = pd.DataFrame(index=df_continuous.index)
for attr in attributes:
    df_clean_raw[attr] = df_stl_wrain[(attr, 'trend')] + df_stl_wrain[(attr, 'seasonal')] + df_stl_wrain[(attr, 'residual')]

# 2. Slice into your balanced 5-year environmental blocks (September to September)
df_early = df_clean_raw.loc['1997-09-07':'2002-09-07', attributes]
df_recent = df_clean_raw.loc['2020-09-07':'2025-09-07', attributes]

# 3. Calculate the Pearson correlation matrices for each era
corr_early = df_early.corr(method='pearson')
corr_recent = df_recent.corr(method='pearson')

# 4. Set up the side-by-side heatmap plot
fig, axes = plt.subplots(2, 1, figsize=(8.27, 9.5), sharex=True)

# Clean LaTeX formatting for the plot labels
display_labels = [r'CDOM', r'Chl-a', r'K$_{d490}$', 'SPM', r"SST", 'Rain', 'Lagged-Rain', 'Wind-Speed', 'Highest-Gust']

cbar_ax = fig.add_axes([0.73, 0.35, 0.015, 0.45])

# Plot 1: Baseline Era (1997-2002)
sns.heatmap(corr_early, ax=axes[0], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, cbar=False, square=True,
            annot_kws={"size": 8.5},
            xticklabels=display_labels, yticklabels=display_labels)
axes[0].set_title('Historical Era: Raw Time Series\n(1997–2002)',
                   fontsize=11, fontweight='normal', pad=15)
axes[0].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[0].tick_params(axis='y', labelrotation=0, labelsize=9)
axes[0].text(1.02, 0.97, '(a)', transform=axes[0].transAxes,
             fontsize=12, fontweight='bold', va='top', ha='left')

# Plot 2: Modern Era (2020-2025)
sns.heatmap(corr_recent, ax=axes[1], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, square=True,
            annot_kws={"size": 8}, cbar_ax = cbar_ax,
            xticklabels=display_labels, yticklabels=display_labels)
axes[1].set_title('Modern Era: Raw Time Series\n(2020–2025)',
                   fontsize=11, fontweight='normal', pad=15)
axes[1].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[1].tick_params(axis='y', labelrotation=0, labelsize=9)
axes[1].text(1.02, 0.97, '(b)', transform=axes[1].transAxes,
             fontsize=12, fontweight='bold', va='top', ha='left')

cbar_ax.set_ylabel(r'Pearson Correlation Coefficient ($\mathit{r}$)', fontsize=9.5)
cbar_ax.tick_params(labelsize=8.5)

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')
        label.set_rotation_mode('anchor')

matrix_center = (0.05 + 0.84)/2
fig.suptitle("Bulk Signal Correlations: Irish Coast", fontsize=12, fontweight='bold', y=0.99, x=matrix_center)
fig.subplots_adjust(left=0.05, right=0.84, top=0.91, bottom=0.10, hspace=0.22)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/era_correlation_comparison.png', dpi=1200, bbox_inches='tight')
plt.show()

# 5. Optional: Generate and display the mathematical shift between eras
corr_diff = corr_recent - corr_early
print("Correlation Change Matrix (Modern minus Baseline):")
print(corr_diff)

Correlation Change Matrix (Modern minus Baseline):
                    cdom     chl_a     kd490       spm     sst_C  sumrain_7d  \
cdom            0.000000  0.157513  0.137133  0.038248  0.086511    0.016277   
chl_a           0.157513  0.000000  0.003689  0.136735 -0.211936    0.007323   
kd490           0.137133  0.003689  0.000000  0.147471 -0.230841   -0.000528   
spm             0.038248  0.136735  0.147471  0.000000 -0.144144   -0.086644   
sst_C           0.086511 -0.211936 -0.230841 -0.144144  0.000000   -0.032338   
sumrain_7d      0.016277  0.007323 -0.000528 -0.086644 -0.032338    0.000000   
lag_sumrain_7d  0.014928  0.038807  0.035591 -0.012510 -0.051904   -0.093005   
meanwind_7d    -0.123339 -0.002887 -0.002868 -0.034886  0.058568   -0.022608   
maxburst_7d    -0.105609 -0.009196 -0.021183 -0.036411  0.062820    0.043968   

                lag_sumrain_7d  meanwind_7d  maxburst_7d  
cdom                  0.014928    -0.123339    -0.105609  
chl_a                 0.038807

In [33]:
df_stl_wrain.xs('trend', level=1, axis=1)

,cdom,chl_a,kd490,lag_sumrain_7d,maxburst_7d,meanwind_7d,spm,sst_C,sumrain_7d
1997-09-07,0.157938,2.185596,0.136175,22.622174,40.130944,11.804609,5.382980,11.680613,21.811456
1997-09-14,0.157919,2.184147,0.136124,22.626401,40.137505,11.806448,5.379907,11.680748,21.819479
1997-09-21,0.157897,2.182686,0.136071,22.631694,40.144809,11.808692,5.376853,11.680839,21.828523
1997-09-28,0.157875,2.181229,0.136018,22.638097,40.152950,11.811337,5.373830,11.680883,21.838612
1997-10-05,0.157852,2.179786,0.135966,22.645683,40.161952,11.814355,5.370844,11.680873,21.849662
...,...,...,...,...,...,...,...,...,...
2025-08-10,0.153258,2.889151,0.156363,18.834292,34.643880,10.732157,2.734159,12.175356,19.237605
2025-08-17,0.153234,2.889505,0.156373,18.848957,34.647801,10.733351,2.734322,12.177704,19.254851
2025-08-24,0.153207,2.889820,0.156383,18.862561,34.651073,10.734477,2.734495,12.179988,19.270472
2025-08-31,0.153178,2.890079,0.156391,18.874919,34.653684,10.735527,2.734683,12.182204,19.284412


In [34]:

# 1. Isolate the long-term trend component across all attributes using your exact syntax
df_all_trends = df_stl_wrain.xs('trend', level=1, axis=1)

# Order attributes clearly for the plot
attributes = ['cdom', 'chl_a', 'kd490', 'spm', 'sst_C', 'sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']
df_all_trends = df_all_trends[attributes]

# 2. Slice the trends into your exact balanced era blocks
df_trend_early = df_all_trends.loc['1997-09-07':'2002-09-07']
df_trend_recent = df_all_trends.loc['2020-09-07':'2025-09-07']

# 3. Calculate the Pearson correlation for the trends within each era
corr_trend_early = df_trend_early.corr()
corr_trend_recent = df_trend_recent.corr()

# 4. Plot them side-by-side
fig, axes = plt.subplots(2, 1, figsize=(8.27, 9.5), sharex=True)
display_labels = [r'CDOM', r'Chl-a', r'K$_{d490}$', 'SPM', r"SST", 'Rain', 'Lagged-Rain', 'Wind-Speed', 'Highest-Gust']

cbar_ax = fig.add_axes([0.73, 0.35, 0.015, 0.45])

# Baseline Trend Matrix
sns.heatmap(corr_trend_early, ax=axes[0], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, cbar=False, square=True,annot_kws={"size": 8},
            xticklabels=display_labels, yticklabels=display_labels)
axes[0].set_title('Historical Era: Trend-Only Time Series\n(1997–2002)',
                   fontsize=11, fontweight='normal', pad=15)
axes[0].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[0].tick_params(axis='y', labelrotation=0, labelsize=9)
axes[0].text(1.02, 0.97, '(a)', transform=axes[0].transAxes,
             fontsize=12, fontweight='bold', va='top', ha='left')
# Modern Trend Matrix
sns.heatmap(corr_trend_recent, ax=axes[1], annot=True, fmt=".3f", cmap="RdBu_r",
            cbar_ax = cbar_ax,annot_kws={"size": 8},
            vmin=-1, vmax=1,  square=True,
            xticklabels=display_labels, yticklabels=display_labels)
axes[1].set_title('Modern Era: Trend-Only Time Series\n(2020–2025)',
                   fontsize=11, fontweight='normal', pad=15)
axes[1].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[1].tick_params(axis='y', labelrotation=0, labelsize=9)
axes[1].text(1.02, 0.97, '(b)', transform=axes[1].transAxes,
             fontsize=12, fontweight='bold', va='top', ha='left')

cbar_ax.set_ylabel(r'Pearson Correlation Coefficient ($\mathit{r}$)', fontsize=9.5)
cbar_ax.tick_params(labelsize=8.5)

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')
        label.set_rotation_mode('anchor')

#fig.suptitle("Environmental Trend Correlation Analysis: Irish Coast", fontsize=12, fontweight='bold', y=0.99)
matrix_center = (0.05 + 0.84) / 2  # This equals 0.445

# Apply the calculated center to the 'x' parameter
fig.suptitle("Isolated Trend Correlations: Irish Coast",
             fontsize=12, fontweight='bold', x=matrix_center, y=0.99)
fig.subplots_adjust(left=0.05, right=0.84, top=0.91, bottom=0.10, hspace=0.22)
plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/era_trend_correlation_comparison.png', dpi=1200, bbox_inches='tight')
plt.show()

# Print out the direct change matrix for your text description
print("Trend Correlation Shift (Modern Trend Matrix minus Baseline Trend Matrix):")
print(corr_trend_recent - corr_trend_early)

Trend Correlation Shift (Modern Trend Matrix minus Baseline Trend Matrix):
                    cdom     chl_a     kd490       spm     sst_C  sumrain_7d  \
cdom            0.000000 -0.620458 -0.658358  0.316865 -0.109592   -0.760818   
chl_a          -0.620458  0.000000 -0.022354  0.997062 -0.908817    0.828142   
kd490          -0.658358 -0.022354  0.000000  0.775863 -1.068500    0.812270   
spm             0.316865  0.997062  0.775863  0.000000  1.157496   -0.309707   
sst_C          -0.109592 -0.908817 -1.068500  1.157496  0.000000   -0.455141   
sumrain_7d     -0.760818  0.828142  0.812270 -0.309707 -0.455141    0.000000   
lag_sumrain_7d -0.737973  0.975246  0.970664 -0.400787 -0.572562   -0.001941   
meanwind_7d     0.764933  1.155321  1.248833 -0.904383  0.034863    0.280460   
maxburst_7d     0.599908  1.411704  1.335223 -0.267941  0.594993    0.282383   

                lag_sumrain_7d  meanwind_7d  maxburst_7d  
cdom                 -0.737973     0.764933     0.599908  
chl_a 

In [35]:

df_stl_wrain

cdom                         chl_a                      \
            residual  seasonal     trend  residual  seasonal     trend   
1997-09-07  0.008799 -0.009435  0.157938  0.049434 -0.260103  2.185596   
1997-09-14 -0.000275 -0.000341  0.157919  0.012337 -0.221557  2.184147   
1997-09-21 -0.004700  0.004105  0.157897  0.034426 -0.242185  2.182686   
1997-09-28 -0.008196 -0.014087  0.157875  0.028412 -0.236212  2.181229   
1997-10-05  0.002444  0.001977  0.157852  0.091016 -0.268021  2.179786   
...              ...       ...       ...       ...       ...       ...   
2025-08-10  0.005282 -0.016920  0.153258 -0.332628 -0.344802  2.889151   
2025-08-17  0.001565 -0.018098  0.153234  0.184354  0.203948  2.889505   
2025-08-24 -0.026916 -0.060104  0.153207 -0.361160 -1.316639  2.889820   
2025-08-31 -0.001627 -0.021005  0.153178 -0.306974 -0.423155  2.890079   
2025-09-07 -0.011446 -0.014658  0.153147  0.267358  0.112397  2.890271   

               kd490                     lag_sumrain_7d  ... meanwind_7d  \
            residual  seasonal     trend       residual  ...       trend   
1997-09-07  0.003639 -0.003790  0.136175       5.673635  ...   11.804609   
1997-09-14  0.002035 -0.002134  0.136124       0.432514  ...   11.806448   
1997-09-21  0.001465 -0.001512  0.136071      -6.622221  ...   11.808692   
1997-09-28  0.001285 -0.003212  0.136018       4.329441  ...   11.811337   
1997-10-05  0.002637 -0.008095  0.135966      -2.940126  ...   11.814355   
...              ...       ...       ...            ...  ...         ...   
2025-08-10 -0.009422 -0.013704  0.156363      -3.727634  ...   10.732157   
2025-08-17  0.007247  0.006583  0.156373      -0.121188  ...   10.733351   
2025-08-24 -0.014858 -0.045517  0.156383      -6.483527  ...   10.734477   
2025-08-31 -0.012368 -0.017698  0.156391      -3.007454  ...   10.735527   
2025-09-07  0.010992  0.005700  0.156396       8.244548  ...   10.736497   

                 spm                         sst_C                       \
            residual  seasonal     trend  residual  seasonal      trend   
1997-09-07  0.200310 -1.551899  5.382980 -0.032741  3.615565  11.680613   
1997-09-14  0.066604 -1.415120  5.379907 -0.026601  3.477270  11.680748   
1997-09-21  0.277498 -1.622961  5.376853 -0.031496  3.271859  11.680839   
1997-09-28  0.517327 -0.053661  5.373830 -0.035752  3.045318  11.680883   
1997-10-05 -0.302675 -1.962214  5.370844 -0.020592  2.787819  11.680873   
...              ...       ...       ...       ...       ...        ...   
2025-08-10 -0.055808 -1.222236  2.734159  0.048416  3.764251  12.175356   
2025-08-17  0.216570 -0.840418  2.734322  0.086191  4.011408  12.177704   
2025-08-24 -0.207788 -1.395685  2.734495  0.040333  3.997477  12.179988   
2025-08-31  0.202293 -0.545506  2.734683 -0.051783  3.839700  12.182204   
2025-09-07 -0.005542 -0.837333  2.734895 -0.060029  3.563640  12.184351   

           sumrain_7d                        
             residual   seasonal      trend  
1997-09-07   0.630715  -0.262171  21.811456  
1997-09-14  -6.487588  -5.491891  21.819479  
1997-09-21   4.463540  -1.672063  21.828523  
1997-09-28  -2.976408 -16.082204  21.838612  
1997-10-05  -8.182197  -4.607466  21.849662  
...               ...        ...        ...  
2025-08-10  -0.288900  -5.111206  19.237605  
2025-08-17  -6.686569 -10.143282  19.254851  
2025-08-24  -3.139918 -13.543054  19.270472  
2025-08-31   8.044819  11.108268  19.284412  
2025-09-07   8.961665   0.541644  19.296692  

[1462 rows x 27 columns]

In [36]:
import matplotlib.pyplot as plt

# 1. Isolate just the trend components from your MultiIndex DataFrame
#attributes = ['kd490', 'chl_a', 'spm', 'sst', 'cdom', 'sumrain_7d', 'lag_sumrain_7d']
attributes = ['kd490', 'chl_a', 'spm','cdom','sst_C','sumrain_7d',  'maxburst_7d'] #'meanwind_7d',
df_trends = df_stl_wrain.xs('trend', level=1, axis=1)

# 2. Initialize a stacked multi-panel plot
fig, axes = plt.subplots(nrows=7, ncols=1, figsize=(12, 10), sharex=True)

# Define distinct oceanographic colors for visual clarity
#colors = ['#1f77b4', '#2ca02c', '#9467bd', '#d62728', '#ff7f0e', '#5be3e1', '#94b7e3' ]
#units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$mg\ L^{-1}$', '°K', r'$m^{-1}$', r'mm $week^{-1}$', r'mm $week^{-1}$']
#labels = ['$K_d(490)$', 'Chlorophyll-a', 'SPM', 'SST', 'CDOM', 'Rain', 'Lagged-Rain']
colors = ['#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e','#d62728',  '#6dacfc',  '#5f5b87'  ]  #'#8b9bb0',
units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$mg\ L^{-1}$', r'$m^{-1}$','°C',  r'mm $week^{-1}$',r'kt'] #, r'kt',
labels = ['$K_d(490)$', 'Chlorophyll-a', 'SPM','CDOM', 'SST',  'Rain',  'Highest Gust'] #'Mean Wind Speed',

for ax, attr, label, unit, color in zip(axes, attributes, labels, units, colors):
    # Plot the 28-year trend line
    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2, label=f'{label} Trend')

## yearly grid marks
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.YearLocator(1))

    # Aesthetics
    ax.set_ylabel(f'{label}\n[{unit}]', fontsize=10, fontweight='normal')
    # vertical grid lines for every year
    ax.grid(True, which='major', axis='x',color='darkgrey', linestyle='--', alpha=0.5)
    ax.grid(True, which='minor', axis='x', color='lightgrey', linestyle = '--', alpha=0.3)
    ## horizontal lines
    ax.grid(True, which='major', axis='y', color='darkgrey', linestyle='--', alpha=0.5)
    #ax.legend(loc='upper left')

# Final formatting
axes[-1].set_xlabel('Year', fontsize=12, fontweight='normal')
plt.suptitle('28-Year Long-Term Trend Trajectories (Irish Coast)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

_Not the final plot_

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

-# =====================================================================
-# 0. GLOBAL DATA DEFINITIONS
-# =====================================================================
attributes = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C', 'sumrain_7d', 'maxburst_7d']
labels = ['$K_d(490)$', 'Chlorophyll-a', 'SPM', 'CDOM', 'SST', 'Rain', 'Highest Gust']
units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$mg\ L^{-1}$', r'$m^{-1}$', '°C', r'mm $week^{-1}$', 'kt']
colors = ['#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e', '#d62728', '#6dacfc', '#5f5b87']
panel_letters = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']

x_start = pd.Timestamp('1997-01-01')
x_end = pd.Timestamp('2025-12-31')

-# Extract just the trend components from your MultiIndex DataFrame
df_trends = df_stl_wrain.xs('trend', level=1, axis=1)

-# =====================================================================
-# 1. PART I: OPTICS & BIOGEOCHEMISTRY (Panels a - d)
-# =====================================================================
part1_indices = [0, 1, 2, 3]

-# Height reduced to 7.5 inches. Leaves ~4.2 inches at the bottom of A4 for a caption.
fig1, axes1 = plt.subplots(nrows=len(part1_indices), ncols=1, figsize=(8.27, 7.5), sharex=True)

for idx_output, idx_global in enumerate(part1_indices):
    ax = axes1[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    color = colors[idx_global]

    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2)

    -# Yearly grid marks
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.YearLocator(1))

    ax.set_xlim(x_start, x_end)
    -# Aesthetics & Clean Axis labels (No internal legends)
    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='x', color='darkgrey', linestyle='--', alpha=0.5)
    ax.grid(True, which='minor', axis='x', color='lightgrey', linestyle='--', alpha=0.3)
    ax.grid(True, which='major', axis='y', color='darkgrey', linestyle='--', alpha=0.5)

    -# Add external panel letters in top-right corner
    ax.text(1.04, 0.90, panel_letters[idx_global], transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='right', verticalalignment='bottom')

-# Part I Global Layout
axes1[-1].set_xlabel('Year', fontsize=10, fontweight='normal')
plt.suptitle('28-Year Long-Term Trend (Irish Coast) - Part I', fontsize=12, fontweight='bold', y=1.00, x=0.60)
-# Strict margin budgets: left=0.24 keeps plots safely right of the 4cm mark
fig1.subplots_adjust(left=0.24, right=0.95, top=0.96, bottom=0.10, hspace=0.38)
-#fig1.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/coast_long_term_trends_part1.png', dpi=1200)


-# =====================================================================
-# 2. PART II: METEOROLOGICAL & METRIC DRIVERS (Panels e - g)
-# =====================================================================
part2_indices = [4, 5, 6]

-# Height reduced to 5.8 inches for 3 panels. Leaves ~5.9 inches for a long caption.
fig2, axes2 = plt.subplots(nrows=len(part2_indices), ncols=1, figsize=(8.27, 5.8), sharex=True)

for idx_output, idx_global in enumerate(part2_indices):
    ax = axes2[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    color = colors[idx_global]

    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2)

    # Yearly grid marks
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.YearLocator(1))

    ax.set_xlim(x_start, x_end)

    -# Aesthetics
    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='x', color='darkgrey', linestyle='--', alpha=0.5)
    ax.grid(True, which='minor', axis='x', color='lightgrey', linestyle='--', alpha=0.3)
    ax.grid(True, which='major', axis='y', color='darkgrey', linestyle='--', alpha=0.5)

    -# Add external panel letters
    ax.text(1.04, 0.90, panel_letters[idx_global], transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='right', verticalalignment='bottom')

-# Part II Global Layout
axes2[-1].set_xlabel('Year', fontsize=10, fontweight='normal')
plt.suptitle('28-Year Long-Term Trend (Irish Coast) - Part II', fontsize=12, fontweight='bold', y=1.00, x=0.60)
-# Left and Right positions perfectly match Part I to ensure vertical layout alignment
fig2.subplots_adjust(left=0.24, right=0.95, top=0.96, bottom=0.10, hspace=0.38)
-#fig2.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/coast_long_term_trends_part2.png', dpi=1200)


plt.show()

In [37]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =====================================================================
# 0. GLOBAL DATA DEFINITIONS
# =====================================================================
attributes = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C', 'sumrain_7d', 'meanwind_7d', 'maxburst_7d']
labels = ['$K_{d490}$', 'Chlorophyll-a', 'SPM', 'CDOM', 'SST', 'Rain', 'Mean Wind Speed', 'Highest Gust']
units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$g\ m^{-3}$', r'$m^{-1}$', '°C', r'mm $week^{-1}$', 'kt','kt']
colors = ['#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e', '#d62728', '#6dacfc', '#8b9bb0','#5f5b87']
panel_letters = ['(a)', '(b)', '(c)', '(d)', '(e)', '(a)', '(b)', '(c)']

x_start = pd.Timestamp('1997-01-01')
x_end = pd.Timestamp('2025-12-31')

# Extract just the trend components from your MultiIndex DataFrame
df_trends = df_stl_wrain.xs('trend', level=1, axis=1)

# =====================================================================
# 1. PART I: OPTICS & BIOGEOCHEMISTRY (Panels a - d)
# =====================================================================
part1_indices = [0, 1, 2, 3, 4]

# Height reduced to 7.5 inches. Leaves ~4.2 inches at the bottom of A4 for a caption.
fig1, axes1 = plt.subplots(nrows=len(part1_indices), ncols=1, figsize=(8.27, 7.5), sharex=True)

for idx_output, idx_global in enumerate(part1_indices):
    ax = axes1[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    color = colors[idx_global]

    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2)

    # Yearly grid marks
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.YearLocator(1))

    ax.set_xlim(x_start, x_end)
    # Aesthetics & Clean Axis labels (No internal legends)
    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='x', color='darkgrey', linestyle='--', alpha=0.5)
    ax.grid(True, which='minor', axis='x', color='lightgrey', linestyle='--', alpha=0.3)
    ax.grid(True, which='major', axis='y', color='darkgrey', linestyle='--', alpha=0.5)

    # Add external panel letters in top-right corner
    ax.text(1.04, 0.90, panel_letters[idx_global], transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='right', verticalalignment='bottom')

# Part I Global Layout
axes1[-1].set_xlabel('Year', fontsize=10, fontweight='normal')
grid_center_1 = (fig1.subplotpars.left + fig1.subplotpars.right) / 2

# This locks the title perfectly to that grid center at a safe height (y=0.96)
fig1.suptitle('28-Year Long-Term Trend (Irish Coast) - Part I',
             fontsize=12, fontweight='bold', x=grid_center_1, y=0.98)
#plt.suptitle('28-Year Long-Term Trend (Irish Coast) - Part I', fontsize=12, fontweight='bold', y=0.98, x=0.54)
# Strict margin budgets: left=0.24 keeps plots safely right of the 4cm mark
fig1.subplots_adjust(left=0.12, right=0.95, top=0.94, bottom=0.10, hspace=0.38)
fig1.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/coast_long_term_trends.png', dpi=1200)


# =====================================================================
# 2. PART II: METEOROLOGICAL & METRIC DRIVERS (Panels e - g)
# =====================================================================
part2_indices = [5, 6, 7]

# Height reduced to 5.8 inches for 3 panels. Leaves ~5.9 inches for a long caption.
fig2, axes2 = plt.subplots(nrows=len(part2_indices), ncols=1, figsize=(8.27, 5.8), sharex=True)

for idx_output, idx_global in enumerate(part2_indices):
    ax = axes2[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    color = colors[idx_global]

    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2)

    # Yearly grid marks
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.YearLocator(1))

    ax.set_xlim(x_start, x_end)

    # Aesthetics
    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='x', color='darkgrey', linestyle='--', alpha=0.5)
    ax.grid(True, which='minor', axis='x', color='lightgrey', linestyle='--', alpha=0.3)
    ax.grid(True, which='major', axis='y', color='darkgrey', linestyle='--', alpha=0.5)

    # Add external panel letters
    ax.text(1.04, 0.90, panel_letters[idx_global], transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='right', verticalalignment='bottom')

# Part II Global Layout
axes2[-1].set_xlabel('Year', fontsize=10, fontweight='normal')
## use this method below for the title, it allows it to be perfectly centered
grid_center_2 = (fig2.subplotpars.left + fig2.subplotpars.right) / 2

# This locks the title perfectly to that grid center at a safe height (y=0.96)
fig2.suptitle('28-Year Long-Term Trend (Atmospheric Variables)',
             fontsize=12, fontweight='bold', x=grid_center_2, y=0.99)
#plt.suptitle('28-Year Long-Term Trend (Atmospheric Variables)', fontsize=12, fontweight='bold', y=0.99) #, x=0.60)
# Left and Right positions perfectly match Part I to ensure vertical layout alignment
fig2.subplots_adjust(left=0.12, right=0.95, top=0.94, bottom=0.10, hspace=0.38)
fig2.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/atmo_long_term_trends.png', dpi=1200)


plt.show()

In [38]:
##

## want open ocean correlation

In [39]:
import pandas as pd
from statsmodels.tsa.seasonal import STL
## Added in SST in Celcius, and then took the Kelvin one out when trends were taken out for plotting
# 1. Extract the raw open ocean satellite data
# (Ensure attributes match lowercase naming conventions)
sat_attributes = ['kd490', 'chl_a', 'spm', 'sst', 'cdom']
df_ocean_sat = ocean_ts_ds[sat_attributes].to_dataframe()
df_ocean_sat['sst_C'] = df_ocean_sat['sst'] - 273.15
sat_attributes = ['kd490', 'chl_a', 'spm', 'sst','sst_C', 'cdom']

### old stl

-# --- FIX: Create a continuous 7-day grid anchored to your true data start date ---
-# This prevents pandas from shifting your natural dates to arbitrary calendar Sundays
start_date = df_ocean_sat.index.min()
end_date = df_ocean_sat.index.max()
continuous_dates = pd.date_range(start=start_date, end=end_date, freq='7D')


-# Reindex using the true continuous 7-day environmental grid
df_ocean_sat = df_ocean_sat.reindex(continuous_dates)

-# 2. Map and append the atmospheric drivers using the same index-matching strategy
-# This ensures weather timelines match the open ocean satellite dates perfectly
atmo_attributes = ['sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']

for col in atmo_attributes:
    # Pull the pre-aligned atmospheric data directly from your coastal/master dataframe
    # since both share the same weekly Sunday-anchored index timeline
    df_ocean_sat[col] = df_continuous[col]

-# Define the full consolidated attribute list for the loop
all_attributes = sat_attributes + atmo_attributes

-# 3. Initialize containers for the decomposed open-ocean elements
ocean_trends = {}
ocean_seasonals = {}
ocean_residuals = {}

print("Beginning Open Ocean MultiIndex STL decomposition...")
for attr in all_attributes:
    print(f" -> Decomposing Open Ocean {attr}...")

    # Interpolate gaps (clouds or edge boundaries)
    filled_series = df_ocean_sat[attr].interpolate(method='time').bfill().ffill()

    # Fit the robust STL model (period=52 for weekly data)
    res = STL(filled_series, period=52, robust=True).fit()

    # Store components
    ocean_trends[attr] = res.trend
    ocean_seasonals[attr] = res.seasonal
    ocean_residuals[attr] = res.resid

-# 4. Consolidate into a clean, matching MultiIndex DataFrame structure
df_ocean_trends_df = pd.DataFrame(ocean_trends, index=df_ocean_sat.index)
df_ocean_seasonals_df = pd.DataFrame(ocean_seasonals, index=df_ocean_sat.index)
df_ocean_residuals_df = pd.DataFrame(ocean_residuals, index=df_ocean_sat.index)

-# Combine using your exact coastal MultiIndex formatting
df_ocean_stl = pd.concat(
    {'trend': df_ocean_trends_df, 'seasonal': df_ocean_seasonals_df, 'residual': df_ocean_residuals_df},
    axis=1
).swaplevel(0, 1, axis=1).sort_index(axis=1)
trend_ocean_correlation = df_ocean_stl.xs('trend', level=1, axis=1).corr()
trend_ocean_correlation.to_csv('satellite_trend_correlation.csv', index=False)
trend_ocean_correlation

print("Open Ocean attributes successfully decomposed into MultiIndex format!")

### new ocean stl with ARIMA

In [40]:
# --- FIX: Create a continuous 7-day grid anchored to your true data start date ---
start_date = df_ocean_sat.index.min()
end_date = df_ocean_sat.index.max()
continuous_dates = pd.date_range(start=start_date, end=end_date, freq='7D')

# Reindex using the true continuous 7-day environmental grid
df_ocean_sat = df_ocean_sat.reindex(continuous_dates)

# 2. Map and append the atmospheric drivers using the same index-matching strategy
atmo_attributes = ['sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']

for col in atmo_attributes:
    df_ocean_sat[col] = df_continuous[col]

# Define the full consolidated attribute list for the loop
all_attributes = sat_attributes + atmo_attributes

# 3. Initialize containers for the decomposed open-ocean elements
ocean_trends = {}
ocean_seasonals = {}
ocean_residuals = {}

# Define padding length: 104 weeks (~2 years on both sides)
pad_weeks = 104

# Automatically extract your exact dataset cadence ('7D')
native_freq = df_ocean_sat.index.freq
if native_freq is None:
    native_freq = pd.infer_freq(df_ocean_sat.index)
    if native_freq is None:
        native_freq = '7D'

print("Beginning Open Ocean Dual-Ended Predictive MultiIndex STL decomposition...")
for attr in all_attributes:
    print(f" -> Decomposing Open Ocean {attr}...")

    # Interpolate gaps (clouds or edge boundaries)
    filled_series = df_ocean_sat[attr].interpolate(method='time').bfill().ffill()

    # Force native explicit frequency mapping onto the individual tracking array
    filled_series.index.freq = native_freq

    # =================================================================
    # A. FORECASTING FOR THE TRAILING EDGE (End of Data)
    # =================================================================
    try:
        model_fwd = sm.tsa.statespace.SARIMAX(
            filled_series, order=(1,0,0), seasonal_order=(0,1,0,52),
            enforce_stationarity=False, enforce_invertibility=False
        )
        fit_fwd = model_fwd.fit(disp=False)
        forecast_raw = fit_fwd.forecast(steps=pad_weeks)

        future_dates = pd.date_range(
            start=end_date + pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        forecast_extended = pd.Series(forecast_raw.values, index=future_dates)
    except:
        future_dates = pd.date_range(
            start=end_date + pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        forecast_extended = pd.Series(np.tile(filled_series.iloc[-52:].values, 2), index=future_dates)

    # =================================================================
    # B. BACKCASTING FOR THE LEADING EDGE (Start of Data)
    # =================================================================
    try:
        # Drop DatetimeIndex and pass raw values to prevent non-monotonic warnings
        reversed_values = filled_series.values[::-1]

        model_bwd = sm.tsa.statespace.SARIMAX(
            reversed_values, order=(1,0,0), seasonal_order=(0,1,0,52),
            enforce_stationarity=False, enforce_invertibility=False
        )
        fit_bwd = model_bwd.fit(disp=False)
        backcast_raw = fit_bwd.forecast(steps=pad_weeks)

        backcast_dates = pd.date_range(
            end=start_date - pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        backcast_extended = pd.Series(backcast_raw.values[::-1], index=backcast_dates)
    except:
        backcast_dates = pd.date_range(
            end=start_date - pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        backcast_extended = pd.Series(np.tile(filled_series.iloc[:52].values, 2), index=backcast_dates)

    # =================================================================
    # C. STITCH, DECOMPOSE AND CROP BOUNDARIES
    # =================================================================
    # Combine everything into a single timeline and enforce index parameters
    extended_series = pd.concat([backcast_extended, filled_series, forecast_extended]).sort_index()
    extended_series.index.freq = native_freq

    # Fit the robust STL model (trend=105 matches your coastal configuration)
    res_extended = STL(extended_series, period=52, trend=105, robust=True).fit()

    # Store components cropped strictly back to original chronological parameters
    ocean_trends[attr] = res_extended.trend.loc[start_date:end_date]
    ocean_seasonals[attr] = res_extended.seasonal.loc[start_date:end_date]
    ocean_residuals[attr] = res_extended.resid.loc[start_date:end_date]

# 4. Consolidate into a clean, matching MultiIndex DataFrame structure
df_ocean_trends_df = pd.DataFrame(ocean_trends, index=df_ocean_sat.index)
df_ocean_seasonals_df = pd.DataFrame(ocean_seasonals, index=df_ocean_sat.index)
df_ocean_residuals_df = pd.DataFrame(ocean_residuals, index=df_ocean_sat.index)

# Combine using your exact coastal MultiIndex formatting
df_ocean_stl = pd.concat(
    {'trend': df_ocean_trends_df, 'seasonal': df_ocean_seasonals_df, 'residual': df_ocean_residuals_df},
    axis=1
).swaplevel(0, 1, axis=1).sort_index(axis=1)

# Generate stabilized correlation output
trend_ocean_correlation = df_ocean_stl.xs('trend', level=1, axis=1).corr()
trend_ocean_correlation.to_csv('satellite_trend_correlation.csv', index=False)

print("Open Ocean attributes successfully decomposed into stabilized MultiIndex format!")
trend_ocean_correlation

Beginning Open Ocean Dual-Ended Predictive MultiIndex STL decomposition...
 -> Decomposing Open Ocean kd490...
 -> Decomposing Open Ocean chl_a...
 -> Decomposing Open Ocean spm...
 -> Decomposing Open Ocean sst...
 -> Decomposing Open Ocean sst_C...


C:\Users\25298423\PycharmProjects\JupyterProject1\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


 -> Decomposing Open Ocean cdom...
 -> Decomposing Open Ocean sumrain_7d...
 -> Decomposing Open Ocean lag_sumrain_7d...
 -> Decomposing Open Ocean meanwind_7d...
 -> Decomposing Open Ocean maxburst_7d...
Open Ocean attributes successfully decomposed into stabilized MultiIndex format!


,cdom,chl_a,kd490,lag_sumrain_7d,maxburst_7d,meanwind_7d,spm,sst,sst_C,sumrain_7d
cdom,1.000000,0.816990,0.861644,0.128439,0.630489,0.268375,0.537664,-0.295570,-0.295570,0.114965
chl_a,0.816990,1.000000,0.992937,0.122079,0.733776,0.426667,0.744858,-0.166348,-0.166348,0.079708
kd490,0.861644,0.992937,1.000000,0.111945,0.720114,0.393202,0.705041,-0.187950,-0.187950,0.074741
lag_sumrain_7d,0.128439,0.122079,0.111945,1.000000,0.418191,0.413697,0.417953,-0.281233,-0.281233,0.993570
maxburst_7d,0.630489,0.733776,0.720114,0.418191,1.000000,0.824322,0.863836,-0.234260,-0.234260,0.394146
meanwind_7d,0.268375,0.426667,0.393202,0.413697,0.824322,1.000000,0.773256,-0.372507,-0.372507,0.390120
spm,0.537664,0.744858,0.705041,0.417953,0.863836,0.773256,1.000000,-0.183560,-0.183560,0.368070
sst,-0.295570,-0.166348,-0.187950,-0.281233,-0.234260,-0.372507,-0.183560,1.000000,1.000000,-0.279297
sst_C,-0.295570,-0.166348,-0.187950,-0.281233,-0.234260,-0.372507,-0.183560,1.000000,1.000000,-0.279297
sumrain_7d,0.114965,0.079708,0.074741,0.993570,0.394146,0.390120,0.368070,-0.279297,-0.279297,1.000000


In [41]:
# Extract the pure trend slice for all attributes
## took out SST Kelvin here for ease with plots later
df_ocean_pure_trends = df_ocean_stl.xs('trend', level=1, axis=1)[[attr for attr in all_attributes if attr != 'sst']]

# Slice into the matching five-year era blocks
df_ocean_trend_early = df_ocean_pure_trends.loc['1997-09-05':'2002-09-04']
df_ocean_trend_recent = df_ocean_pure_trends.loc['2020-09-05':'2025-09-05']

# Generate the correlation matrices
corr_ocean_early = df_ocean_trend_early.corr()
corr_ocean_recent = df_ocean_trend_recent.corr()

# Save out to CSV for safety
corr_ocean_recent.to_csv('open_ocean_modern_trend_correlation.csv')

In [42]:
a = ['mango', 'banana', 'apple'].remove('apple')
print(a)

None


In [43]:

# Define your exact target oceanographic attributes in order
attributes = ['cdom', 'chl_a', 'kd490', 'spm', 'sst_C', 'sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']

# 1. Reconstruct the clean, cloud-filled continuous data from your STL components
# (This ensures we use the exact data that went into your trends, minus data gaps)
df_clean_raw = pd.DataFrame(index=df_continuous.index)
for attr in attributes:
    df_clean_raw[attr] = df_ocean_stl[(attr, 'trend')] +df_ocean_stl[(attr, 'seasonal')] + df_ocean_stl[
        (attr, 'residual')]

# 2. Slice into your balanced 5-year environmental blocks (September to September)
df_early = df_clean_raw.loc['1997-09-07':'2002-09-07', attributes]
df_recent = df_clean_raw.loc['2020-09-07':'2025-09-07', attributes]

# 3. Calculate the Pearson correlation matrices for each era
corr_early = df_early.corr(method='pearson')
corr_recent = df_recent.corr(method='pearson')

# 4. Set up the side-by-side heatmap plot
fig, axes = plt.subplots(2, 1, figsize=(8.27, 9.5), sharex=True)

# Clean LaTeX formatting for the plot labels
display_labels = [r'CDOM', r'Chl-a', r'K$_{d490}$', 'SPM', r"SST", 'Rain', 'Lagged-Rain', 'Wind-Speed', 'Highest-Gust']

cbar_ax = fig.add_axes([0.73, 0.35, 0.015, 0.45])
# Plot 1: Baseline Era (1997-2002)
sns.heatmap(corr_early, ax=axes[0], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, cbar=False, square=True,annot_kws={"size": 8},
            xticklabels=display_labels, yticklabels=display_labels)
axes[0].set_title('Historical Era: Raw Time Series\n(1997–2002)',
                  fontsize=11, fontweight='normal', pad=15)
axes[0].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[0].tick_params(axis='y', labelrotation=0, labelsize=9)

# Plot 2: Modern Era (2020-2025)
sns.heatmap(corr_recent, ax=axes[1], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, cbar_ax=cbar_ax, square=True,annot_kws={"size": 8},
            xticklabels=display_labels, yticklabels=display_labels)
axes[1].set_title('Modern Era: Raw Time Series\n(2020–2025)',
                  fontsize=11, fontweight='normal', pad=15)
axes[1].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[1].tick_params(axis='y', labelrotation=0, labelsize=9)


cbar_ax.set_ylabel(r'Pearson Correlation Coefficient ($\mathit{r}$)', fontsize=9.5)

cbar_ax.tick_params(labelsize=8.5)


for ax in axes:
   for label in ax.get_xticklabels():
       label.set_horizontalalignment('right')
       label.set_rotation_mode('anchor')

matrix_center = (0.05+0.84)/2
fig.suptitle("Bulk Signal Correlations: Open Ocean", fontsize=12, fontweight='bold', y=0.99, x=matrix_center)
fig.subplots_adjust(left=0.05, right=0.84, top=0.91, bottom=0.10, hspace=0.22)

plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/ocean_era_correlation_comparison.png', dpi=1200, bbox_inches='tight')
plt.show()

# 5. Optional: Generate and display the mathematical shift between eras
corr_diff = corr_recent - corr_early
print("Correlation Change Matrix (Modern minus Baseline):")
print(corr_diff)

Correlation Change Matrix (Modern minus Baseline):
                    cdom     chl_a     kd490       spm     sst_C  sumrain_7d  \
cdom            0.000000  0.009115  0.019100  0.246876 -0.021728    0.074333   
chl_a           0.009115  0.000000  0.010670 -0.068918 -0.040054    0.047869   
kd490           0.019100  0.010670  0.000000 -0.103055 -0.040414    0.057484   
spm             0.246876 -0.068918 -0.103055  0.000000 -0.017926   -0.227283   
sst_C          -0.021728 -0.040054 -0.040414 -0.017926  0.000000   -0.010800   
sumrain_7d      0.074333  0.047869  0.057484 -0.227283 -0.010800    0.000000   
lag_sumrain_7d  0.101571 -0.009828 -0.005826 -0.114470 -0.039045   -0.093005   
meanwind_7d     0.086953  0.012112  0.034499 -0.040220  0.066316   -0.022608   
maxburst_7d     0.092412  0.064868  0.085299 -0.079592  0.069140    0.043968   

                lag_sumrain_7d  meanwind_7d  maxburst_7d  
cdom                  0.101571     0.086953     0.092412  
chl_a                -0.009828

In [44]:
attributes = ['cdom', 'chl_a', 'kd490', 'spm', 'sst_C', 'sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']

# 1. Reconstruct the clean, cloud-filled continuous data from your STL components
# (This ensures we use the exact data that went into your trends, minus data gaps)
df_clean_raw = pd.DataFrame(index=df_continuous.index)
for attr in attributes:
    df_clean_raw[attr] = df_ocean_stl[(attr, 'trend')]

# 2. Slice into your balanced 5-year environmental blocks (September to September)
df_early = df_clean_raw.loc['1997-09-07':'2002-09-07', attributes]
df_recent = df_clean_raw.loc['2020-09-07':'2025-09-07', attributes]

# 3. Calculate the Pearson correlation matrices for each era
corr_early = df_early.corr(method='pearson')
corr_recent = df_recent.corr(method='pearson')
# 4. Plot them side-by-side
fig, axes = plt.subplots(2, 1, figsize=(8.27, 9.5), sharex=True)

display_labels = [r'CDOM', r'Chl-a', r'K$_{d490}$', 'SPM', "SST", 'Rain', 'Lagged-Rain', 'Wind-Speed', 'Highest-Gust']
cbar_ax = fig.add_axes([0.73, 0.35, 0.015, 0.45])
# Baseline Trend Matrix
sns.heatmap(corr_early, ax=axes[0], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, cbar=False, square=True,annot_kws={"size": 8},
            xticklabels=display_labels, yticklabels=display_labels)
axes[0].set_title('Historical Era: Trend-Only Time Series\n(1997–2002)',
                   fontsize=11, fontweight='normal', pad=15)
axes[0].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[0].tick_params(axis='y', labelrotation=0, labelsize=9)
axes[0].text(1.02, 0.97, '(a)', transform=axes[0].transAxes,
            fontsize=12, fontweight='bold', va='top', ha='left')


# Modern Trend Matrix
sns.heatmap(corr_recent, ax=axes[1], annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, cbar_ax=cbar_ax, square=True,annot_kws={"size": 8},
            xticklabels=display_labels, yticklabels=display_labels)
axes[1].set_title('Modern Era: Trend-Only Time Series\n(2020–2025)',
                  fontsize=11, fontweight='normal', pad=15)
axes[1].tick_params(axis='x', labelrotation=35, labelsize=9)
axes[1].tick_params(axis='y', labelrotation=0, labelsize=9)
axes[1].text(1.02, 0.97, '(b)', transform=axes[1].transAxes,
            fontsize=12, fontweight='bold', va='top', ha='left')


cbar_ax.set_ylabel(r'Pearson Correlation Coefficient ($\mathit{r}$)', fontsize=9.5)
cbar_ax.tick_params(labelsize=8.5)


for ax in axes:
   for label in ax.get_xticklabels():
       label.set_horizontalalignment('right')
       label.set_rotation_mode('anchor')

matrix_center = (0.05+0.84)/2
fig.suptitle("Isolated Trend Correlations: Open Ocean", fontsize=12, fontweight='bold', y=0.99, x=matrix_center)
fig.subplots_adjust(left=0.05, right=0.84, top=0.91, bottom=0.10, hspace=0.22)

plt.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/ocean_era_trend_correlation_comparison_.png', dpi=1200, bbox_inches='tight')
plt.show()

# Print out the direct change matrix for your text description
print("Trend Correlation Shift - Open Ocean - (Modern Trend Matrix minus Baseline Trend Matrix):")
print(corr_trend_recent - corr_trend_early)

Trend Correlation Shift - Open Ocean - (Modern Trend Matrix minus Baseline Trend Matrix):
                    cdom     chl_a     kd490       spm     sst_C  sumrain_7d  \
cdom            0.000000 -0.620458 -0.658358  0.316865 -0.109592   -0.760818   
chl_a          -0.620458  0.000000 -0.022354  0.997062 -0.908817    0.828142   
kd490          -0.658358 -0.022354  0.000000  0.775863 -1.068500    0.812270   
spm             0.316865  0.997062  0.775863  0.000000  1.157496   -0.309707   
sst_C          -0.109592 -0.908817 -1.068500  1.157496  0.000000   -0.455141   
sumrain_7d     -0.760818  0.828142  0.812270 -0.309707 -0.455141    0.000000   
lag_sumrain_7d -0.737973  0.975246  0.970664 -0.400787 -0.572562   -0.001941   
meanwind_7d     0.764933  1.155321  1.248833 -0.904383  0.034863    0.280460   
maxburst_7d     0.599908  1.411704  1.335223 -0.267941  0.594993    0.282383   

                lag_sumrain_7d  meanwind_7d  maxburst_7d  
cdom                 -0.737973     0.764933     0.

In [45]:
import matplotlib.pyplot as plt

# 1. Isolate just the trend components from your MultiIndex DataFrame
#attributes = ['kd490', 'chl_a', 'spm', 'sst', 'cdom', 'sumrain_7d', 'lag_sumrain_7d']
attributes = ['kd490', 'chl_a', 'spm','cdom','sst_C','sumrain_7d', 'meanwind_7d', 'maxburst_7d']
df_trends = df_ocean_stl.xs('trend', level=1, axis=1)

# 2. Initialize a stacked multi-panel plot
fig, axes = plt.subplots(nrows=8, ncols=1, figsize=(14, 12), sharex=True)

# Define distinct oceanographic colors for visual clarity
#colors = ['#1f77b4', '#2ca02c', '#9467bd', '#d62728', '#ff7f0e', '#5be3e1', '#94b7e3' ]
#units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$mg\ L^{-1}$', '°K', r'$m^{-1}$', r'mm $week^{-1}$', r'mm $week^{-1}$']
#labels = ['$K_d(490)$', 'Chlorophyll-a', 'SPM', 'SST', 'CDOM', 'Rain', 'Lagged-Rain']
colors = ['#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e','#d62728',  '#6dacfc', '#8b9bb0', '#5f5b87'  ]
units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$g\ m^{-3}$', r'$m^{-1}$','°C',  r'mm $week^{-1}$', r'kt',r'kt']
labels = [r'K$_{d490}$', 'Chlorophyll-a', 'SPM','CDOM', 'SST',  'Rain', 'Mean Wind Speed', 'Highest Gust']

for ax, attr, label, unit, color in zip(axes, attributes, labels, units, colors):
    # Plot the 28-year trend line
    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2, label=f'{label} Trend')

    # Aesthetics
    ax.set_ylabel(f'{label}\n[{unit}]', fontsize=10, fontweight='normal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')

# Final formatting
axes[-1].set_xlabel('Year', fontsize=12, fontweight='normal')
plt.suptitle('28-Year Long-Term Trend Trajectories (Atlantic/Celtic Open Ocean)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [46]:
import matplotlib
import matplotlib.pyplot as plt

import matplotlib.dates as mdates

# =====================================================================
# 0. GLOBAL DATA DEFINITIONS
# =====================================================================
attributes = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C', ]
labels = ['K$_{d490}$', 'Chlorophyll-a', 'SPM', 'CDOM', 'SST']
units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$g\ m^{-3}$', r'$m^{-1}$', '$°C$']
colors = ['#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e', '#d62728']
panel_letters = ['(a)', '(b)', '(c)', '(d)', '(e)']

x_start = pd.Timestamp('1997-01-01')
x_end = pd.Timestamp('2025-12-31')

# Extract just the trend components from your MultiIndex DataFrame
df_trends = df_ocean_stl.xs('trend', level=1, axis=1)

# =====================================================================
# 1. PART I: OPTICS & BIOGEOCHEMISTRY (Panels a - d)
# =====================================================================
part1_indices = [0, 1, 2, 3,4]

# Height reduced to 7.5 inches. Leaves ~4.2 inches at the bottom of A4 for a caption.
fig1, axes1 = plt.subplots(nrows=len(part1_indices), ncols=1, figsize=(8.27, 9.5), sharex=True)

for idx_output, idx_global in enumerate(part1_indices):
    ax = axes1[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    color = colors[idx_global]

    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2)

    # Yearly grid marks
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.YearLocator(1))

    ax.set_xlim(x_start, x_end)
    # Aesthetics & Clean Axis labels (No internal legends)
    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='x', color='darkgrey', linestyle='--', alpha=0.5)
    ax.grid(True, which='minor', axis='x', color='lightgrey', linestyle='--', alpha=0.3)
    ax.grid(True, which='major', axis='y', color='darkgrey', linestyle='--', alpha=0.5)

    # Add external panel letters in top-right corner
    ax.text(1.04, 0.90, panel_letters[idx_global], transform=ax.transAxes, fontsize=10, fontweight='bold',
            horizontalalignment='right', verticalalignment='bottom')

# Part I Global Layout
axes1[-1].set_xlabel('Year', fontsize=10, fontweight='normal')

#fig1.suptitle('28-Year Long-Term Trend (Atlantic/Celtic Open Ocean)', fontsize=12, fontweight='bold', y=1.00, x=0.60)
grid_center_1 = (fig1.subplotpars.left + fig1.subplotpars.right) / 2
fig1.suptitle('28-Year Long-Term Trend (Atlantic/Celtic Open Ocean)',
             fontsize=12, fontweight='bold', x=grid_center_1, y=0.98)
# Strict margin budgets: left=0.24 keeps plots safely right of the 4cm mark
fig1.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.08, hspace=0.38)
fig1.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/ocean_long_term_trends.png', dpi=1200)
plt.show()

## want to double check that the sst is right again - it wasnt! lots of edge interference, fixed with ARIMA and applied above

In [ ]:
sst_all = coastal_ts_ds['sst'].to_dataframe()

In [ ]:
sst_all['sst_C'] = sst_all['sst'] - 273.25
sst_all.head()

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=sst_all, x='time', y='sst_C')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from statsmodels.tsa.seasonal import STL

# =====================================================================
# 1. PREPARE THE TRUNCATED DATASETS & RUN STL
# =====================================================================
# We will test three scenarios to observe how the edge window behaves:
#   Scenario A: The Full Dataset (through Dec 2025)
#   Scenario B: Truncated to Mid-2025 (June 2025)
#   Scenario C: Truncated to Late 2024 (December 2024)

# Ensure the index is a datetime index for proper interpolation and slicing
df_continuous.index = pd.to_datetime(df_continuous.index)

# Define truncation end-dates
cutoffs = {
    'Full Data (Dec 2025)': df_continuous.index.max(),
    'Truncated to Mid-2025': '2025-06-30',
    'Truncated to Late 2024': '2024-12-31'
}

trends_sst_variants = {}

for label, cutoff_date in cutoffs.items():
    # 1. Slice the dataframe to the designated cutoff
    df_sliced = df_continuous.loc[:cutoff_date].copy()

    # 2. Interpolate gaps exactly like your original pipeline
    filled_sst = df_sliced['sst_C'].interpolate(method='time')
    filled_sst = filled_sst.bfill().ffill()

    # 3. Fit the robust STL model (period=52 for weekly data)
    res = STL(filled_sst, period=52, robust=True).fit()

    # 4. Save the trend component
    trends_sst_variants[label] = res.trend

# =====================================================================
# 2. GENERATE JOURNAL-READY COMPARISON PLOT
# =====================================================================
# Sized perfectly for A4 width (8.27 in) with plenty of caption room below (height 5.0 in)
fig, ax = plt.subplots(figsize=(8.27, 5.0))

# Plot colors and styles to distinguish the variants
styles = {
    'Full Data (Dec 2025)': {'color': '#d62728', 'linestyle': '-', 'linewidth': 2.5},
    'Truncated to Mid-2025': {'color': '#2ca02c', 'linestyle': '--', 'linewidth': 2.0},
    'Truncated to Late 2024': {'color': '#1f77b4', 'linestyle': ':', 'linewidth': 2.0}
}

# Plot each trend component variant
for label, trend_series in trends_sst_variants.items():
    ax.plot(trend_series.index, trend_series.values, label=label, **styles[label])

# Formatting the axes to standard publication design
ax.set_ylabel(r'$\Delta$ SST Trend (°C)', fontsize=10, fontweight='normal')
ax.set_xlabel('Year', fontsize=11, fontweight='bold', labelpad=10)

# Configure grids
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.7)
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.4)

# Set X-Axis view range to isolate the suspicious 2023–2025 zone clearly
ax.set_xlim(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-01-01'))

# Place the legend symmetrically balanced within our subplots boundary
ax.legend(loc='upper left', frameon=True, facecolor='white', fontsize=10)

# Panel marker pinned to the inside upper right
ax.text(0.98, 0.96, '(e)', transform=ax.transAxes, fontsize=11, fontweight='bold',
        horizontalalignment='right', verticalalignment='top')

# Precision canvas tuning:
# left=0.193 satisfies the 4cm left margin requirement on an A4 page.
# bottom=0.22 reserves a clean canvas gap for multi-line figure text.
fig.subplots_adjust(left=0.193, right=0.93, top=0.92, bottom=0.22)

# Export at high-fidelity DPI
fig.savefig('sst_trend_edge_verification.png', dpi=1200)
plt.show()

In [ ]:
## the SLM is having issues around the edges, need to find a way to fix this

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from scipy.signal import butter, filtfilt

# =====================================================================
# 1. DEFINE FUNCTIONS AND SCENARIOS
# =====================================================================
def lowpass_trend(series, period=52, cutoff_years=2):
    """Zero-Phase Low-Pass Filter Trend Extractor"""
    fs = period
    cutoff_hz = 1.0 / cutoff_years
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_hz / nyquist

    b, a = butter(N=2, Wn=normal_cutoff, btype='low', analog=False)
    # Using method='gust' optimizes the padding at the absolute edges
    return filtfilt(b, a, series, method='gust')

# Ensure DateTime Index
df_continuous.index = pd.to_datetime(df_continuous.index)

cutoffs = {
    'Full Data (Dec 2025)': df_continuous.index.max(),
    'Truncated to Late 2024': '2024-12-31'
}

results = {}

for label, cutoff_date in cutoffs.items():
    # Slice and clean data identically
    df_sliced = df_continuous.loc[:cutoff_date].copy()
    filled_sst = df_sliced['sst_C'].interpolate(method='time').bfill().ffill()

    # Method 1: Original Robust STL
    res_stl = STL(filled_sst, period=52, robust=True).fit()

    # Method 2: Zero-Phase Butterworth Lowpass Filter
    trend_butter = lowpass_trend(filled_sst, period=52, cutoff_years=2)

    results[label] = {
        'STL': res_stl.trend,
        'Butterworth': pd.Series(trend_butter, index=df_sliced.index)
    }

# =====================================================================
# 2. GENERATE COMPLEMENTARY VERIFICATION PLOTS
# =====================================================================
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(8.27, 8.5), sharex=True)

# ---------------------------------------------------------------------
# PANEL 1: Original Robust STL Framework (The Benchmark)
# ---------------------------------------------------------------------
ax1.plot(results['Full Data (Dec 2025)']['STL'].index,
         results['Full Data (Dec 2025)']['STL'].values,
         color='#d62728', linestyle='-', linewidth=2.5, label='Full Data (Dec 2025)')

ax1.plot(results['Truncated to Late 2024']['STL'].index,
         results['Truncated to Late 2024']['STL'].values,
         color='#1f77b4', linestyle=':', linewidth=2.5, label='Truncated to Late 2024')

ax1.set_ylabel(r'$\Delta$ SST STL Trend (°C)', fontsize=10)
ax1.set_title('Original Framework: Robust STL (Exhibits Edge Pull)', fontsize=11, fontweight='bold', loc='left')
ax1.grid(True, which='major', color='#d3d3d3', linestyle='-', alpha=0.7)
ax1.legend(loc='upper left', frameon=True, facecolor='white', fontsize=9)
ax1.text(0.98, 0.95, '(a)', transform=ax1.transAxes, fontsize=11, fontweight='bold', ha='right', va='top')

# ---------------------------------------------------------------------
# PANEL 2: Zero-Phase Butterworth Digital Filter (The Proposed Change)
# ---------------------------------------------------------------------
ax2.plot(results['Full Data (Dec 2025)']['Butterworth'].index,
         results['Full Data (Dec 2025)']['Butterworth'].values,
         color='#2ca02c', linestyle='-', linewidth=2.5, label='Full Data (Dec 2025)')

ax2.plot(results['Truncated to Late 2024']['Butterworth'].index,
         results['Truncated to Late 2024']['Butterworth'].values,
         color='#bcbd22', linestyle=':', linewidth=2.5, label='Truncated to Late 2024')

ax2.set_ylabel(r'$\Delta$ SST Filtered Trend (°C)', fontsize=10)
ax2.set_xlabel('Year', fontsize=11, fontweight='bold', labelpad=10)
ax2.set_title('Proposed Change: Zero-Phase Lowpass Filter (Stabilized Edges)', fontsize=11, fontweight='bold', loc='left')
ax2.grid(True, which='major', color='#d3d3d3', linestyle='-', alpha=0.7)
ax2.legend(loc='upper left', frameon=True, facecolor='white', fontsize=9)
ax2.text(0.98, 0.95, '(b)', transform=ax2.transAxes, fontsize=11, fontweight='bold', ha='right', va='top')

# Limit focus to the suspicious timeline block
plt.xlim(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-01-01'))

# A4 Sizing Layout Configurations
fig.subplots_adjust(left=0.193, right=0.93, top=0.93, bottom=0.15, hspace=0.3)
fig.savefig('sst_method_comparison_test.png', dpi=1200)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL

# Ensure DateTime Index
df_continuous.index = pd.to_datetime(df_continuous.index)

cutoffs = {
    'Full Data (Dec 2025)': df_continuous.index.max(),
    'Truncated to Late 2024': '2024-12-31'
}

results_hp = {}

for label, cutoff_date in cutoffs.items():
    # Slice and clean data identically
    df_sliced = df_continuous.loc[:cutoff_date].copy()
    filled_sst = df_sliced['sst_C'].interpolate(method='time').bfill().ffill()

    # Method 1: Original Robust STL
    res_stl = STL(filled_sst, period=52, robust=True).fit()

    # Method 2: Hodrick-Prescott Filter
    # lambda=14400 is standard for weekly/monthly long-term macroeconomic & climate trends
    cycle, trend_hp = sm.tsa.filters.hpfilter(filled_sst, lamb=14400)

    results_hp[label] = {
        'STL': res_stl.trend,
        'HP': trend_hp
    }

# =====================================================================
# GENERATE COMPARISON PLOTS
# =====================================================================
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(8.27, 8.5), sharex=True)

# PANEL 1: Original Robust STL
ax1.plot(results_hp['Full Data (Dec 2025)']['STL'].index, results_hp['Full Data (Dec 2025)']['STL'].values,
         color='#d62728', linestyle='-', linewidth=2.5, label='Full Data (Dec 2025)')
ax1.plot(results_hp['Truncated to Late 2024']['STL'].index, results_hp['Truncated to Late 2024']['STL'].values,
         color='#1f77b4', linestyle=':', linewidth=2.5, label='Truncated to Late 2024')
ax1.set_ylabel(r'$\Delta$ SST STL Trend (°C)', fontsize=10)
ax1.set_title('Original Framework: Robust STL (Exhibits Edge Pull)', fontsize=11, fontweight='bold', loc='left')
ax1.grid(True, which='major', color='#d3d3d3', linestyle='-', alpha=0.7)
ax1.legend(loc='upper left', frameon=True, facecolor='white', fontsize=9)
ax1.text(0.98, 0.95, '(a)', transform=ax1.transAxes, fontsize=11, fontweight='bold', ha='right', va='top')

# PANEL 2: Hodrick-Prescott Filter
ax2.plot(results_hp['Full Data (Dec 2025)']['HP'].index, results_hp['Full Data (Dec 2025)']['HP'].values,
         color='#2ca02c', linestyle='-', linewidth=2.5, label='Full Data (Dec 2025)')
ax2.plot(results_hp['Truncated to Late 2024']['HP'].index, results_hp['Truncated to Late 2024']['HP'].values,
         color='#bcbd22', linestyle=':', linewidth=2.5, label='Truncated to Late 2024')
ax2.set_ylabel(r'$\Delta$ SST HP Trend (°C)', fontsize=10)
ax2.set_xlabel('Year', fontsize=11, fontweight='bold', labelpad=10)
ax2.set_title('Proposed Alternative: Hodrick-Prescott Filter (No Seasonality, Stable Edges)', fontsize=11, fontweight='bold', loc='left')
ax2.grid(True, which='major', color='#d3d3d3', linestyle='-', alpha=0.7)
ax2.legend(loc='upper left', frameon=True, facecolor='white', fontsize=9)
ax2.text(0.98, 0.95, '(b)', transform=ax2.transAxes, fontsize=11, fontweight='bold', ha='right', va='top')

plt.xlim(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-01-01'))
fig.subplots_adjust(left=0.193, right=0.93, top=0.93, bottom=0.15, hspace=0.3)
fig.savefig('sst_hp_comparison_test.png', dpi=1200)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

# Ensure DateTime Index
df_continuous.index = pd.to_datetime(df_continuous.index)

# Setup our test boundaries
cutoffs = {
    'Full Data (Dec 2025)': df_continuous.index.max(),
    'Truncated to Late 2024': '2024-12-31'
}

results_padded = {}
pad_weeks = 104  # 2 years of symmetric reflection padding
time_delta = df_continuous.index[1] - df_continuous.index[0]  # Expected weekly interval

for label, cutoff_date in cutoffs.items():
    # 1. Slice and clean the dataset
    df_sliced = df_continuous.loc[:cutoff_date].copy()
    raw_series = df_sliced['sst_C'].interpolate(method='time').bfill().ffill()

    # 2. Extract the tail for reflection
    tail_data = raw_series.iloc[-pad_weeks:].values

    # 3. Create the point-reflected mirror padding
    # (The multiplication by 2 ensures a seamless, continuous hinge point)
    reflected_tail = 2 * raw_series.iloc[-1] - tail_data[::-1]

    # 4. Create pseudo-future timestamps for the padding window
    slice_end_date = df_sliced.index.max()
    future_dates = [slice_end_date + (i * time_delta) for i in range(1, pad_weeks + 1)]
    padded_tail_series = pd.Series(reflected_tail, index=future_dates)

    # 5. Concatenate together to form the extended series
    extended_series = pd.concat([raw_series, padded_tail_series])

    # 6. Run Robust STL on the extended data (trend=105 for a broad 2-year window)
    res_extended = STL(extended_series, period=52, trend=105, robust=True).fit()

    # 7. Crop out the padding to save only the true historical timeline
    results_padded[label] = res_extended.trend.loc[:slice_end_date]

# =====================================================================
# GENERATE THE VERIFICATION GRAPH
# =====================================================================
fig, ax = plt.subplots(figsize=(8.27, 5.0))

# Plot the Full Data line using the new padding pipeline
ax.plot(results_padded['Full Data (Dec 2025)'].index,
        results_padded['Full Data (Dec 2025)'].values,
        color='#2ca02c', linestyle='-', linewidth=2.5, label='Full Data (Dec 2025) with Reflection')

# Plot the Truncated line using the same padding pipeline
ax.plot(results_padded['Truncated to Late 2024'].index,
        results_padded['Truncated to Late 2024'].values,
        color='#bcbd22', linestyle=':', linewidth=2.5, label='Truncated to Late 2024 with Reflection')

# Layout and formatting
ax.set_ylabel(r'$\Delta$ SST Stabilized Trend (°C)', fontsize=10)
ax.set_xlabel('Year', fontsize=11, fontweight='bold', labelpad=10)
ax.grid(True, which='major', color='#d3d3d3', linestyle='-', alpha=0.7)
ax.legend(loc='upper left', frameon=True, facecolor='white', fontsize=10)

# Limit focus to our diagnostic window
plt.xlim(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-01-01'))

# Precise A4 page fitting
fig.subplots_adjust(left=0.193, right=0.93, top=0.92, bottom=0.18)
fig.savefig('sst_reflection_verification_test.png', dpi=1200)

In [ ]:
## still bad, need to try something else

## trying Structural ARIMA Forecast Padding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL

# Ensure DateTime Index
df_continuous.index = pd.to_datetime(df_continuous.index)

cutoffs = {
    'Full Data (Dec 2025)': df_continuous.index.max(),
    'Truncated to Late 2024': '2024-12-31'
}

results_forecast_padded = {}
pad_weeks = 104

for label, cutoff_date in cutoffs.items():
    # 1. Slice and isolate the dataset variant
    df_sliced = df_continuous.loc[:cutoff_date].copy()
    raw_series = df_sliced['sst_C'].interpolate(method='time').bfill().ffill()

    # 2. Fit a quick structural seasonal model to project pseudo-future cycles
    # (1,0,0)x(0,1,0)[52] captures baseline momentum + strict annual seasonality
    try:
        model = sm.tsa.statespace.SARIMAX(
            raw_series,
            order=(1, 0, 0),
            seasonal_order=(0, 1, 0, 52),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        model_fit = model.fit(disp=False)
        forecast_extended = model_fit.forecast(steps=pad_weeks)
    except:
        # Fallback to simple seasonal naive projection if matrix optimization strains
        historical_cycle = raw_series.iloc[-52:]
        forecast_extended = pd.Series(
            np.tile(historical_cycle.values, 2),
            index=pd.date_range(start=raw_series.index[-1] + pd.Timedelta(weeks=1), periods=pad_weeks, freq='W')
        )

    # 3. Append the forecasted future to the historical timeline
    extended_series = pd.concat([raw_series, forecast_extended])

    # 4. Run your exact Robust STL parameter configurations
    res_extended = STL(extended_series, period=52, trend=105, robust=True).fit()

    # 5. Trim off the pseudo-future to preserve your exact 28-year layout
    slice_end_date = df_sliced.index.max()
    results_forecast_padded[label] = res_extended.trend.loc[:slice_end_date]

# =====================================================================
# GENERATE THE VERIFICATION GRAPH
# =====================================================================
fig, ax = plt.subplots(figsize=(8.27, 5.0))

ax.plot(results_forecast_padded['Full Data (Dec 2025)'].index,
        results_forecast_padded['Full Data (Dec 2025)'].values,
        color='#2ca02c', linestyle='-', linewidth=2.5, label='Full Data (Dec 2025) with Forecast Padding')

ax.plot(results_forecast_padded['Truncated to Late 2024'].index,
        results_forecast_padded['Truncated to Late 2024'].values,
        color='#bcbd22', linestyle=':', linewidth=2.5, label='Truncated to Late 2024 with Forecast Padding')

ax.set_ylabel(r'$\Delta$ SST Stabilized Trend (°C)', fontsize=10)
ax.set_xlabel('Year', fontsize=11, fontweight='bold', labelpad=10)
ax.grid(True, which='major', color='#d3d3d3', linestyle='-', alpha=0.7)
ax.legend(loc='upper left', frameon=True, facecolor='white', fontsize=10)

plt.xlim(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-01-01'))
fig.subplots_adjust(left=0.193, right=0.93, top=0.92, bottom=0.18)
fig.savefig('sst_forecast_padding_verification.png', dpi=1200)
plt.show()

In [ ]:
## looks better than before, going to apply it to the beginnning of the time series as well

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
attributes = ['cdom', 'chl_a', 'kd490', 'spm', 'sst_C', 'sumrain_7d', 'lag_sumrain_7d', 'meanwind_7d', 'maxburst_7d']
# Ensure DataFrame Index is a clean DatetimeIndex
df_continuous.index = pd.to_datetime(df_continuous.index)
original_start_date = df_continuous.index.min()
original_end_date = df_continuous.index.max()

# Define padding length: 104 weeks (~2 years on both sides)
pad_weeks = 104

trends = {}
seasonals = {}
residuals = {}

print("Beginning Dual-Ended Predictive STL Decomposition...")

# Automatically extract your exact dataset cadence (e.g., '7D')
native_freq = df_continuous.index.freq
if native_freq is None:
    native_freq = pd.infer_freq(df_continuous.index)
    if native_freq is None:
        # Fallback to your explicit generation step if inference struggles
        native_freq = '7D'

for attr in attributes:
    print(f" -> Decomposing {attr}...")

    # 1. Clean the raw data slice
    raw_series = df_continuous[attr].interpolate(method='time').bfill().ffill()

    # Marry the inferred/extracted explicit frequency to the series index
    raw_series.index.freq = native_freq

    # =================================================================
    # A. FORECASTING FOR THE TRAILING EDGE (End of Data)
    # =================================================================
    try:
        model_fwd = sm.tsa.statespace.SARIMAX(
            raw_series, order=(1,0,0), seasonal_order=(0,1,0,52),
            enforce_stationarity=False, enforce_invertibility=False
        )
        fit_fwd = model_fwd.fit(disp=False)
        forecast_raw = fit_fwd.forecast(steps=pad_weeks)

        # Build future dates building cleanly off your exact end timestamp
        future_dates = pd.date_range(
            start=original_end_date + pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        forecast_extended = pd.Series(forecast_raw.values, index=future_dates)
    except:
        # Robust structural fallback matched to your exact step frequency
        future_dates = pd.date_range(
            start=original_end_date + pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        forecast_extended = pd.Series(np.tile(raw_series.iloc[-52:].values, 2), index=future_dates)

    # =================================================================
    # B. BACKCASTING FOR THE LEADING EDGE (Start of Data)
    # =================================================================
    try:
        # Drop the DatetimeIndex and convert to raw values before reversing to avoid warnings
        reversed_values = raw_series.values[::-1]

        model_bwd = sm.tsa.statespace.SARIMAX(
            reversed_values, order=(1,0,0), seasonal_order=(0,1,0,52),
            enforce_stationarity=False, enforce_invertibility=False
        )
        fit_bwd = model_bwd.fit(disp=False)
        backcast_raw = fit_bwd.forecast(steps=pad_weeks)

        # Build historical dates stepping cleanly backward from your start timestamp
        backcast_dates = pd.date_range(
            end=original_start_date - pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        backcast_extended = pd.Series(backcast_raw.values[::-1], index=backcast_dates)
    except:
        # Robust structural fallback matched to your exact step frequency
        backcast_dates = pd.date_range(
            end=original_start_date - pd.to_timedelta(native_freq),
            periods=pad_weeks,
            freq=native_freq
        )
        backcast_extended = pd.Series(np.tile(raw_series.iloc[:52].values, 2), index=backcast_dates)

    # =================================================================
    # C. STITCH AND DECOMPOSE
    # =================================================================
    # Merge everything into a single timeline and ensure a consistent step pattern
    extended_series = pd.concat([backcast_extended, raw_series, forecast_extended]).sort_index()
    extended_series.index.freq = native_freq

    # Fit the Robust STL on the stabilized extended dataframe
    res_extended = STL(extended_series, period=52, trend=105, robust=True).fit()

    # Crop out both padding ends to restore the exact original 28-year dimensions
    trends[attr] = res_extended.trend.loc[original_start_date:original_end_date]
    seasonals[attr] = res_extended.seasonal.loc[original_start_date:original_end_date]
    residuals[attr] = res_extended.resid.loc[original_start_date:original_end_date]

# =====================================================================
# 2. CONSOLIDATE INTO MULTI-INDEX STRUCTURING
# =====================================================================
df_trends = pd.DataFrame(trends, index=df_continuous.index)
df_seasonals = pd.DataFrame(seasonals, index=df_continuous.index)
df_residuals = pd.DataFrame(residuals, index=df_continuous.index)

df_stl_wrain = pd.concat(
    {'trend': df_trends, 'seasonal': df_seasonals, 'residual': df_residuals},
    axis=1
).swaplevel(0, 1, axis=1).sort_index(axis=1)

print("Dual-ended stabilization complete! Calculating matrices...")
trend_correlation = df_stl_wrain.xs('trend', level=1, axis=1).corr()
trend_correlation.to_csv('satellite_trend_correlation.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

# 1. Isolate just the trend components from your MultiIndex DataFrame
#attributes = ['kd490', 'chl_a', 'spm', 'sst', 'cdom', 'sumrain_7d', 'lag_sumrain_7d']
attributes = [ 'kd490','chl_a', 'spm','cdom', 'sst_C', 'sumrain_7d',  'meanwind_7d', 'maxburst_7d']
df_trends = df_stl_wrain.xs('trend', level=1, axis=1)

# 2. Initialize a stacked multi-panel plot
fig, axes = plt.subplots(nrows=8, ncols=1, figsize=(14, 12), sharex=True)

# Define distinct oceanographic colors for visual clarity
#colors = ['#1f77b4', '#2ca02c', '#9467bd', '#d62728', '#ff7f0e', '#5be3e1', '#94b7e3' ]
#units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$mg\ L^{-1}$', '°K', r'$m^{-1}$', r'mm $week^{-1}$', r'mm $week^{-1}$']
#labels = ['$K_d(490)$', 'Chlorophyll-a', 'SPM', 'SST', 'CDOM', 'Rain', 'Lagged-Rain']
colors = ['#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e','#d62728',  '#6dacfc', '#8b9bb0', '#5f5b87'  ]
units = [r'$m^{-1}$', r'$mg\ m^{-3}$', r'$mg\ L^{-1}$', r'$m^{-1}$','°C',  r'mm $week^{-1}$', r'kt',r'kt']
labels = ['$K_d(490)$', 'Chlorophyll-a', 'SPM','CDOM', 'SST',  'Rain', 'Mean Wind Speed', 'Highest Gust']

for ax, attr, label, unit, color in zip(axes, attributes, labels, units, colors):
    # Plot the 28-year trend line
    ax.plot(df_trends.index, df_trends[attr], color=color, linewidth=2, label=f'{label} Trend')

    # Aesthetics
    ax.set_ylabel(f'{label}\n[{unit}]', fontsize=10, fontweight='normal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')

# Final formatting
axes[-1].set_xlabel('Year', fontsize=12, fontweight='normal')
plt.suptitle('28-Year Long-Term Trend Trajectories (with ARIMA forecasting padding)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

need to re run everything with this kind of analysis

## Seasonality

In [ ]:
true_weeks = df_continuous.groupby(df_continuous.index.month).apply(lambda x: x.index.min().isocalendar().week)
true_weeks_list = true_weeks.tolist()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 1. Isolate the seasonal component from your MultiIndex DataFrame
chl_seasonal = df_stl_wrain[('chl_a', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
chl_seasonal['Week'] = chl_seasonal.index.isocalendar().week
chl_seasonal = chl_seasonal[chl_seasonal['Week'] <= 52]

# 3. Slice the exact 5-year chunks using full September-to-September spans
early_block = chl_seasonal.loc['1997-09-07':'2002-09-06']
recent_block = chl_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the raw average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean().copy()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean().copy()

# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#aec7e8', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#2ca02c', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    # Pulls cleanly from df_continuous as verified
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

month_week_positions.append(52)
month_labels.append('End')

ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting
plt.title('Chlorophyll-a Seasonal Phenology Shift (Preserved Peak Amplitude)', fontsize=12, fontweight='bold')
plt.xlabel('Calendar Week of the Year', fontsize=11, fontweight='bold')
plt.ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
plt.xlim(1, 52)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Isolate the seasonal component from your MultiIndex DataFrame
sst_seasonal = df_stl_wrain[('sst_C', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
sst_seasonal['Week'] = sst_seasonal.index.isocalendar().week
sst_seasonal = sst_seasonal[sst_seasonal['Week'] <= 52]  # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
# Early Era: Sept 5, 1997 to Sept 4, 2002 (5 full environmental years)
early_block = sst_seasonal.loc['1997-09-07':'2002-09-06']

# Modern Era: Sept 5, 2020 to Sept 5, 2025 (5 full environmental years)
recent_block = sst_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean()

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#f7bae2', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#d62728', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# 1. Map every single date in the master index to its respective Month and ISO week
df_weeks = pd.DataFrame({
    'Month': df_continuous.index.month,
    'Week': df_continuous.index.isocalendar().week
})

# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 3. Define the short text codes for your months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 4. Use a list comprehension to dynamically build the formatted f-string labels
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# 5. Append the closing barrier manually (Week 52 represents the end of the timeline)
month_week_positions.append(52)
month_labels.append('End')

# 6. Cleanly apply them to your axes
ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)

# Add minor ticks on every individual week
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
# 1. Stronger grid lines for the main month divisions
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)

# 2. Ultra-light, thin grid lines running on every individual week
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)
# Formatting to show the full calendar year on the X-axis
plt.title('Sea Surface Temperature Seasonal Shift (Balanced Environmental Years)', fontsize=12, fontweight='bold')
plt.xlabel('Calendar Week of the Year', fontsize=11, fontweight='bold')
plt.ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
plt.xlim(1, 52)
#plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# DIAGNOSTIC INVESTIGATION: Inspecting Individual Years in Modern Block
# =====================================================================
# Create a clean copy of the modern block to check raw trends
investigate_df = sst_seasonal.loc['2020-09-07':'2025-09-06'].copy()
investigate_df['Year'] = investigate_df.index.year

print("--- MISSING DATA REPORT (Modern Era) ---")
print(investigate_df.isna().sum())

plt.figure(figsize=(12, 6))
# Plot each individual year's raw seasonal track before averaging
for year, group in investigate_df.groupby('Year'):
    # Group by week within that specific year
    weekly_track = group.groupby('Week')['seasonal_value'].mean()
    plt.plot(weekly_track.index, weekly_track.values, label=f'Year {year}', alpha=0.7, linewidth=1.5)

plt.title("SST Seasonal Component Broken Down by Individual Years", fontsize=12, fontweight='bold')
plt.xlabel("Week of Year")
plt.ylabel("Seasonal Amplitude")
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
raw_sst_window = df_continuous.loc['2023':'2025', 'sst']
print(f"Actual missing weeks in raw data: {raw_sst_window.isna().sum()}")

# Print specifically where the 2024 gaps are
print("\n--- Raw Missing Weeks in 2024 ---")
print(raw_sst_window[raw_sst_window.isna() & (raw_sst_window.index.year == 2024)].index.strftime('%Y-%m-%d').tolist())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 1. Isolate the seasonal component from your MultiIndex DataFrame
sst_seasonal = df_stl_wrain[('sst_C', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
sst_seasonal['Week'] = sst_seasonal.index.isocalendar().week
sst_seasonal = sst_seasonal[sst_seasonal['Week'] <= 52]  # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
early_block = sst_seasonal.loc['1997-09-07':'2002-09-06']
recent_block = sst_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the raw average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean().copy()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean().copy()

# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#f7bae2', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#d62728', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

month_week_positions.append(52)
month_labels.append('End')

ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting to show the full calendar year on the X-axis
plt.title('Sea Surface Temperature Seasonal Shift (Preserved Peak Amplitude)', fontsize=12, fontweight='bold')
plt.xlabel('Calendar Week of the Year', fontsize=11, fontweight='bold')
plt.ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
plt.xlim(1, 52)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# 1. Isolate the seasonal component from your MultiIndex DataFrame
cdom_seasonal = df_stl_wrain[('cdom', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
cdom_seasonal['Week'] = cdom_seasonal.index.isocalendar().week
cdom_seasonal = cdom_seasonal[cdom_seasonal['Week'] <= 52]  # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
# Early Era: Sept 5, 1997 to Sept 4, 2002 (5 full environmental years)
early_block = cdom_seasonal.loc['1997-09-07':'2002-09-06']

# Modern Era: Sept 5, 2020 to Sept 5, 2025 (5 full environmental years)
recent_block = cdom_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean()

# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#f5c87f', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#d68f1c', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 3. Define the short text codes for your months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 4. Use a list comprehension to dynamically build the formatted f-string labels
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# 5. Append the closing barrier manually (Week 52 represents the end of the timeline)
month_week_positions.append(52)
month_labels.append('End')

# 6. Cleanly apply them to your axes
ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)

# Add minor ticks on every individual week
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
# 1. Stronger grid lines for the main month divisions
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)

# 2. Ultra-light, thin grid lines running on every individual week
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting to show the full calendar year on the X-axis
plt.title('CDOM Seasonal Shift (Balanced Environmental Years)', fontsize=12, fontweight='bold')
plt.xlabel('Calendar Week of the Year', fontsize=11, fontweight='bold')
plt.ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
plt.xlim(1, 52)
#plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 1. Isolate the seasonal component from your MultiIndex DataFrame
spm_seasonal = df_stl_wrain[('spm', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
spm_seasonal['Week'] = spm_seasonal.index.isocalendar().week
spm_seasonal = spm_seasonal[spm_seasonal['Week'] <= 52] # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
# Early Era: Sept 5, 1997 to Sept 4, 2002 (5 full environmental years)
early_block = spm_seasonal.loc['1997-09-07':'2002-09-06']

# Modern Era: Sept 5, 2020 to Sept 5, 2025 (5 full environmental years)
recent_block = spm_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean()






# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#be86f7', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#460587', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 3. Define the short text codes for your months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 4. Use a list comprehension to dynamically build the formatted f-string labels
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# 5. Append the closing barrier manually (Week 52 represents the end of the timeline)
month_week_positions.append(52)
month_labels.append('End')

# 6. Cleanly apply them to your axes
ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)

# Add minor ticks on every individual week
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
# 1. Stronger grid lines for the main month divisions
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)

# 2. Ultra-light, thin grid lines running on every individual week
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting
ax.set_title('SPM Seasonal Phenology Shift (Balanced Environmental Years)', fontsize=12, fontweight='bold')
ax.set_xlabel('Timeline (Calendar Week & Month)', fontsize=11, fontweight='bold')
ax.set_ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
ax.set_xlim(1, 52)

#plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
df_stl_wrain

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 1. Isolate the seasonal component from your MultiIndex DataFrame
rain_seasonal = df_stl_wrain[('sumrain_7d', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
rain_seasonal['Week'] = rain_seasonal.index.isocalendar().week
rain_seasonal = rain_seasonal[rain_seasonal['Week'] <= 52] # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
# Early Era: Sept 5, 1997 to Sept 4, 2002 (5 full environmental years)
early_block = rain_seasonal.loc['1997-09-07':'2002-09-06']

# Modern Era: Sept 5, 2020 to Sept 5, 2025 (5 full environmental years)
recent_block = rain_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean()






# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#b0c2d9', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#6dacfc', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 3. Define the short text codes for your months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 4. Use a list comprehension to dynamically build the formatted f-string labels
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# 5. Append the closing barrier manually (Week 52 represents the end of the timeline)
month_week_positions.append(52)
month_labels.append('End')

# 6. Cleanly apply them to your axes
ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)

# Add minor ticks on every individual week
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
# 1. Stronger grid lines for the main month divisions
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)

# 2. Ultra-light, thin grid lines running on every individual week
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting
ax.set_title('Rainfall Seasonal Phenology Shift (Balanced Environmental Years)', fontsize=12, fontweight='bold')
ax.set_xlabel('Timeline (Calendar Week & Month)', fontsize=11, fontweight='bold')
ax.set_ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
ax.set_xlim(1, 52)

#plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 1. Isolate the seasonal component from your MultiIndex DataFrame
wind_seasonal = df_stl_wrain[('meanwind_7d', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
wind_seasonal['Week'] = wind_seasonal.index.isocalendar().week
wind_seasonal = wind_seasonal[wind_seasonal['Week'] <= 52] # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
# Early Era: Sept 5, 1997 to Sept 4, 2002 (5 full environmental years)
early_block = wind_seasonal.loc['1997-09-07':'2002-09-06']

# Modern Era: Sept 5, 2020 to Sept 5, 2025 (5 full environmental years)
recent_block = wind_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean()






# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#c3c6c9', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#8b9bb0', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 3. Define the short text codes for your months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 4. Use a list comprehension to dynamically build the formatted f-string labels
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# 5. Append the closing barrier manually (Week 52 represents the end of the timeline)
month_week_positions.append(52)
month_labels.append('End')

# 6. Cleanly apply them to your axes
ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)

# Add minor ticks on every individual week
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
# 1. Stronger grid lines for the main month divisions
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)

# 2. Ultra-light, thin grid lines running on every individual week
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting
ax.set_title('Windspeed Seasonal Phenology Shift (Balanced Environmental Years)', fontsize=12, fontweight='bold')
ax.set_xlabel('Timeline (Calendar Week & Month)', fontsize=11, fontweight='bold')
ax.set_ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
ax.set_xlim(1, 52)

#plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 1. Isolate the seasonal component from your MultiIndex DataFrame
gust_seasonal = df_stl_wrain[('maxburst_7d', 'seasonal')].to_frame(name='seasonal_value')

# 2. Add the calendar week column for the X-axis grouping
gust_seasonal['Week'] = gust_seasonal.index.isocalendar().week
gust_seasonal = gust_seasonal[gust_seasonal['Week'] <= 52] # Remove rare week 53 anomalies

# 3. Slice the exact 5-year chunks using full September-to-September spans
# Early Era: Sept 5, 1997 to Sept 4, 2002 (5 full environmental years)
early_block = gust_seasonal.loc['1997-09-07':'2002-09-06']

# Modern Era: Sept 5, 2020 to Sept 5, 2025 (5 full environmental years)
recent_block = gust_seasonal.loc['2020-09-07':'2025-09-06']

# 4. Group by calendar week to get the average annual cycle for each era
early_era_avg = early_block.groupby('Week')['seasonal_value'].mean()
recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean()






# --- THE MICRO-PIXEL BRIDGE: Only matches the absolute seam, leaving peaks alone ---
for era in [early_era_avg, recent_era_avg]:
    mid_winter_value = (era.loc[1] + era.loc[52]) / 2
    era.loc[1] = mid_winter_value
    era.loc[52] = mid_winter_value

# 5. Plot the annual cycle comparison
plt.figure(figsize=(11, 5.5))
plt.plot(early_era_avg.index, early_era_avg.values, color='#bbb8d4', linewidth=2,
         label='Baseline Era (1997–2002 Environmental Years)')
plt.plot(recent_era_avg.index, recent_era_avg.values, color='#5f5b87', linewidth=2.5,
         label='Modern Era (2020–2025 Environmental Years)')
ax = plt.gca()

# --- Robust Full-Timeline Month Anchors ---
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)


# 2. Find the absolute first week that appears for each month across all years,
# then take the median value to establish the definitive seasonal anchor.
month_week_positions = []
for m in range(1, 13):
    # Find the minimum week that registered for this month in each distinct calendar year
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(lambda x: int(x.min().isocalendar().week))
    # Take the median of those minimum weeks to eliminate anomalies
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

# 3. Define the short text codes for your months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 4. Use a list comprehension to dynamically build the formatted f-string labels
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# 5. Append the closing barrier manually (Week 52 represents the end of the timeline)
month_week_positions.append(52)
month_labels.append('End')

# 6. Cleanly apply them to your axes
ax.set_xticks(month_week_positions)
ax.set_xticklabels(month_labels, fontsize=9)

# Add minor ticks on every individual week
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# --- Enhanced Dual-Grid Hierarchy ---
# 1. Stronger grid lines for the main month divisions
ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.8)

# 2. Ultra-light, thin grid lines running on every individual week
ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.5)

# Formatting
ax.set_title('Highest Gust Seasonal Phenology Shift (Balanced Environmental Years)', fontsize=12, fontweight='bold')
ax.set_xlabel('Timeline (Calendar Week & Month)', fontsize=11, fontweight='bold')
ax.set_ylabel('Seasonal Component Amplitude', fontsize=11, fontweight='bold')
ax.set_xlim(1, 52)

#plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# =====================================================================
# 1. PRE-COMPUTE COMMON X-AXIS MONTH ANCHORS (Done once to save memory)
# =====================================================================
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

# Append the closing barrier
month_week_positions.append(52)
month_labels.append('End')

# =====================================================================
# 2. DEFINE PLOT METADATA FOR THE LOOP
# =====================================================================
attributes = ['chl_a', 'cdom', 'spm','sst_C', 'sumrain_7d', 'meanwind_7d', 'maxburst_7d']
labels = ['Chl-a',  'CDOM', 'SPM','SST', 'Rainfall', 'Windspeed', 'Highest Gust']
units = [r'mg m$^{-3}$ amplitude',  r' m$^{-1}$ amplitude', r' g m$^{-3}$ amplitude',r'°C amplitude', r'mm week$^{-1}$ amplitude', r'kt amplitude', r'kt amplitude']
panel_letters = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']
# Pair up your custom early vs recent colors for each panel
early_colors = ['#aec7e8', '#f5c87f', '#be86f7','#f7bae2', '#b0c2d9', '#c3c6c9', '#bbb8d4']
recent_colors = ['#2ca02c',  '#d68f1c', '#460587','#d62728', '#6dacfc', '#8b9bb0', '#5f5b87']

# =====================================================================
# 3. INITIALIZE MULTI-PANEL CANVAS
# =====================================================================
fig, axes = plt.subplots(nrows=7, ncols=1, figsize=(10, 22), sharex=True)

# Loop through each attribute and its matching subplot axis
for i, (ax, attr, label, unit, letter, c_early, c_recent) in enumerate(zip(axes, attributes, labels, units, panel_letters, early_colors, recent_colors)):

    # Isolate seasonal component
    seasonal_df = df_stl_wrain[(attr, 'seasonal')].to_frame(name='seasonal_value')
    seasonal_df['Week'] = seasonal_df.index.isocalendar().week
    seasonal_df = seasonal_df[seasonal_df['Week'] <= 52]

    # Slice historical vs modern environmental blocks
    early_block = seasonal_df.loc['1997-09-07':'2002-09-06']
    recent_block = seasonal_df.loc['2020-09-07':'2025-09-06']

    # Calculate era means
    early_era_avg = early_block.groupby('Week')['seasonal_value'].mean().copy()
    recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean().copy()

    # Apply the Micro-Pixel Seam Bridge to wrap the year smoothly
    for era in [early_era_avg, recent_era_avg]:
        mid_winter_value = (era.loc[1] + era.loc[52]) / 2
        era.loc[1] = mid_winter_value
        era.loc[52] = mid_winter_value

    # Plot the lines
    ax.plot(early_era_avg.index, early_era_avg.values, color=c_early, linewidth=2,
            label='Baseline Era (1997–2002)', linestyle = '--')
    ax.plot(recent_era_avg.index, recent_era_avg.values, color=c_recent, linewidth=2.5,
            label='Modern Era (2020–2025)', linestyle='-')

    # Y-Axis Aesthetics
    ax.set_ylabel(unit, fontsize=10, fontweight='normal')

    # Grid Setup (Major on months, minor on individual weeks)
    ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.7)
    ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.4)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    # Add a clean, text-based identifier inside each panel instead of full sub-legends
    ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=10, fontweight='bold',
            verticalalignment='bottom', horizontalalignment='left')
    ax.text(1.01, 0.97, letter, transform=ax.transAxes, fontsize=10, fontweight='bold',
            horizontalalignment='left', verticalalignment='top')
# =====================================================================
# 4. FINAL GLOBAL FORMATTING (Applied strictly to the bottom axis)
# =====================================================================
axes[-1].set_xticks(month_week_positions)
axes[-1].set_xticklabels(month_labels, fontsize=9.5)
axes[-1].set_xlabel('Time of Year', fontsize=12, fontweight='bold')
plt.xlim(1, 52)

legend_elements = [
    Line2D([0], [0], color='#888888', linewidth=2, linestyle='--', label='Baseline Era (1997–2002)'),
    Line2D([0], [0], color='#888888', linewidth=2.5, linestyle='-', label='Modern Era (2020–2025)')
]

fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.965), ncol=2, frameon=True, facecolor='white', fontsize=11)

plt.suptitle('Coastal Seasonal Phenology Shifts (Irish Coast)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
# Adjust layout to make sure the global legend doesn't overlap the top plot
plt.subplots_adjust(top=0.94)
plt.show()

## Final Seasonality Plots

In [47]:
## the final one!

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# =====================================================================
# 1. PRE-COMPUTE COMMON X-AXIS MONTH ANCHORS
# =====================================================================
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

month_week_positions.append(52)
month_labels.append('End')

# =====================================================================
# 2. GLOBAL METADATA DEFINTIONS
# =====================================================================
attributes = ['kd490','chl_a',  'spm','cdom', 'sst_C', 'sumrain_7d', 'meanwind_7d', 'maxburst_7d']
labels = [r'$\Delta$ K$_{d490}$', r'$\Delta$ Chl-a', r'$\Delta$ SPM', r'$\Delta$ CDOM', r'$\Delta$ SST', r'$\Delta$ Rainfall',r'$\Delta$ Windspeed', r'$\Delta$ Highest Gust']
units = [r'm$^{-1}$', r'mg m$^{-3}$',  r' g m$^{-3}$', r' m$^{-1}$',r'°C', r'mm week$^{-1}$', r'kt', r'kt']
panel_letters = ['(a)', '(b)', '(c)', '(d)', '(e)', '(a)','(b)','(c)']

early_colors = ['#a4ceeb','#aec7e8',  '#be86f7','#f5c87f', '#f7bae2', '#b0c2d9', '#c3c6c9', '#bbb8d4']
recent_colors = ['#1f77b4','#2ca02c',  '#460587','#d68f1c', '#d62728', '#6dacfc', '#8b9bb0', '#5f5b87']



legend_elements = [
    Line2D([0], [0], color='#888888', linewidth=2, linestyle='--', label='Historical Era (1997–2002)'),
    Line2D([0], [0], color='#888888', linewidth=2.5, linestyle='-', label='Modern Era (2020–2025)')
]

# =====================================================================
# 3. GENERATE PART I: WATER OPTICS & MARINE METRICS (Panels a - d)
# =====================================================================
part1_indices = [0, 1, 2, 3, 4]
fig1, axes1 = plt.subplots(nrows=len(part1_indices), ncols=1, figsize=(8.27, 10.0), sharex=True)

for idx_output, idx_global in enumerate(part1_indices):
    ax = axes1[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    letter = panel_letters[idx_global]
    c_early = early_colors[idx_global]
    c_recent = recent_colors[idx_global]

    # Isolate seasonal component
    seasonal_df = df_stl_wrain[(attr, 'seasonal')].to_frame(name='seasonal_value')
    seasonal_df['Week'] = seasonal_df.index.isocalendar().week
    seasonal_df = seasonal_df[seasonal_df['Week'] <= 52]

    # Slice blocks
    early_block = seasonal_df.loc['1997-09-07':'2002-09-06']
    recent_block = seasonal_df.loc['2020-09-07':'2025-09-06']

    # Era Means
    early_era_avg = early_block.groupby('Week')['seasonal_value'].mean().copy()
    recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean().copy()

    # Micro-Pixel Seam Bridge
    for era in [early_era_avg, recent_era_avg]:
        mid_winter_value = (era.loc[1] + era.loc[52]) / 2
        era.loc[1] = mid_winter_value
        era.loc[52] = mid_winter_value

    # Plotting
    ax.plot(early_era_avg.index, early_era_avg.values, color=c_early, linewidth=2, linestyle='--')
    ax.plot(recent_era_avg.index, recent_era_avg.values, color=c_recent, linewidth=2.5, linestyle='-')

    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.7)
    ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.7)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    # Position External Text Targets
    #ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=10, fontweight='bold', verticalalignment='bottom', horizontalalignment='left')
    ax.text(1.01, 0.97, letter, transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='left', verticalalignment='top')

# Part I Global Format
axes1[-1].set_xticks(month_week_positions)
axes1[-1].set_xticklabels(month_labels, fontsize=9.5, rotation=90, ha='center')
axes1[-1].set_xlabel('Time of Year', fontsize=11, fontweight='normal', labelpad=15)
plt.xlim(1, 52)

fig1.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5615, 0.93), ncol=2, frameon=True, facecolor='white', fontsize=10)
fig1.suptitle("Coastal Seasonality and Environmental Drivers (Irish Coast)", fontsize=12, fontweight='bold', x=0.5615, y=0.96)

fig1.tight_layout()
fig1.subplots_adjust(left = 0.193, right = 0.93, top=0.87, bottom=0.15, hspace=0.22)
fig1.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/coast_seasonality_shifts.png', dpi=1200)

# =====================================================================
# 4. GENERATE PART II: METEOROLOGICAL FORCING (Panels e - g)
# =====================================================================
part2_indices = [ 5, 6, 7]
fig2, axes2 = plt.subplots(nrows=len(part2_indices), ncols=1, figsize=(8.27, 7.8), sharex=True)

for idx_output, idx_global in enumerate(part2_indices):
    ax = axes2[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    letter = panel_letters[idx_global]
    c_early = early_colors[idx_global]
    c_recent = recent_colors[idx_global]

    # Isolate seasonal component
    seasonal_df = df_stl_wrain[(attr, 'seasonal')].to_frame(name='seasonal_value')
    seasonal_df['Week'] = seasonal_df.index.isocalendar().week
    seasonal_df = seasonal_df[seasonal_df['Week'] <= 52]

    # Slice blocks
    early_block = seasonal_df.loc['1997-09-07':'2002-09-06']
    recent_block = seasonal_df.loc['2020-09-07':'2025-09-06']

    # Era Means
    early_era_avg = early_block.groupby('Week')['seasonal_value'].mean().copy()
    recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean().copy()

    # Micro-Pixel Seam Bridge
    for era in [early_era_avg, recent_era_avg]:
        mid_winter_value = (era.loc[1] + era.loc[52]) / 2
        era.loc[1] = mid_winter_value
        era.loc[52] = mid_winter_value

    # Plotting
    ax.plot(early_era_avg.index, early_era_avg.values, color=c_early, linewidth=2, linestyle='--')
    ax.plot(recent_era_avg.index, recent_era_avg.values, color=c_recent, linewidth=2.5, linestyle='-')

    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.7)
    ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.7)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    # Position External Text Targets
    #ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=10, fontweight='bold', verticalalignment='bottom', horizontalalignment='left')
    ax.text(1.01, 0.97, letter, transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='left', verticalalignment='top')

# Part II Global Format
axes2[-1].set_xticks(month_week_positions)
axes2[-1].set_xticklabels(month_labels, fontsize=9.5, rotation=90, ha='center')
axes2[-1].set_xlabel('Time of Year', fontsize=11, fontweight='normal', labelpad=15)
plt.xlim(1, 52)

fig2.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5615, 0.93), ncol=2, frameon=True, facecolor='white', fontsize=10)
fig2.suptitle('Atmospheric Seasonality and Environmental Drivers', fontsize=12, fontweight='bold', x=0.5615, y=0.96)

fig2.tight_layout()
fig2.subplots_adjust(left = 0.193, right = 0.93, top=0.85, bottom=0.18, hspace=0.22)
fig2.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/atmo_seasonality_shifts.png', dpi=1200)

plt.show()

### open ocean seasonality

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# =====================================================================
# 1. PRE-COMPUTE COMMON X-AXIS MONTH ANCHORS
# =====================================================================
month_week_positions = []
for m in range(1, 13):
    annual_min_weeks = df_continuous[df_continuous.index.month == m].index.to_series().groupby(lambda x: x.year).apply(
        lambda x: int(x.min().isocalendar().week))
    median_start_week = int(annual_min_weeks.median())
    month_week_positions.append(median_start_week)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = [f"{name} (W{week})" for name, week in zip(month_names, month_week_positions)]

month_week_positions.append(52)
month_labels.append('End')

# =====================================================================
# 2. GLOBAL METADATA DEFINTIONS
# =====================================================================
attributes = ['kd490','chl_a',  'spm','cdom', 'sst_C']
labels = [r'$\Delta$ K$_{d490}$', r'$\Delta$ Chl-a', r'$\Delta$ SPM', r'$\Delta$ CDOM', r'$\Delta$ SST']
units = [r'm$^{-1}$', r'mg m$^{-3}$',  r' g m$^{-3}$', r' m$^{-1}$',r'°C']
panel_letters = ['(a)', '(b)', '(c)', '(d)', '(e)']

early_colors = ['#a4ceeb','#aec7e8',  '#be86f7','#f5c87f', '#f7bae2']
recent_colors = ['#1f77b4','#2ca02c',  '#460587','#d68f1c', '#d62728']

legend_elements = [
    Line2D([0], [0], color='#888888', linewidth=2, linestyle='--', label='Historical Era (1997–2002)'),
    Line2D([0], [0], color='#888888', linewidth=2.5, linestyle='-', label='Modern Era (2020–2025)')
]

# =====================================================================
# 3. GENERATE PART I: WATER OPTICS & MARINE METRICS (Panels a - d)
# =====================================================================
part1_indices = [0, 1, 2, 3, 4]
fig1, axes1 = plt.subplots(nrows=len(part1_indices), ncols=1, figsize=(8.27, 10.0), sharex=True)

for idx_output, idx_global in enumerate(part1_indices):
    ax = axes1[idx_output]
    attr = attributes[idx_global]
    label = labels[idx_global]
    unit = units[idx_global]
    letter = panel_letters[idx_global]
    c_early = early_colors[idx_global]
    c_recent = recent_colors[idx_global]

    # Isolate seasonal component
    seasonal_df = df_ocean_stl[(attr, 'seasonal')].to_frame(name='seasonal_value')
    seasonal_df['Week'] = seasonal_df.index.isocalendar().week
    seasonal_df = seasonal_df[seasonal_df['Week'] <= 52]

    # Slice blocks
    early_block = seasonal_df.loc['1997-09-07':'2002-09-06']
    recent_block = seasonal_df.loc['2020-09-07':'2025-09-06']

    # Era Means
    early_era_avg = early_block.groupby('Week')['seasonal_value'].mean().copy()
    recent_era_avg = recent_block.groupby('Week')['seasonal_value'].mean().copy()

    # Micro-Pixel Seam Bridge
    for era in [early_era_avg, recent_era_avg]:
        mid_winter_value = (era.loc[1] + era.loc[52]) / 2
        era.loc[1] = mid_winter_value
        era.loc[52] = mid_winter_value

    # Plotting
    ax.plot(early_era_avg.index, early_era_avg.values, color=c_early, linewidth=2, linestyle='--')
    ax.plot(recent_era_avg.index, recent_era_avg.values, color=c_recent, linewidth=2.5, linestyle='-')

    ax.set_ylabel(f'{label}\n({unit})', fontsize=10, fontweight='normal')
    ax.grid(True, which='major', axis='both', color='#d3d3d3', linestyle='-', alpha=0.7)
    ax.grid(True, which='minor', axis='x', color='#e0e0e0', linestyle=':', alpha=0.7)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    # Position External Text Targets
    #ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=10, fontweight='bold', verticalalignment='bottom', horizontalalignment='left')
    ax.text(1.01, 0.97, letter, transform=ax.transAxes, fontsize=10, fontweight='bold', horizontalalignment='left', verticalalignment='top')

# Part I Global Format
axes1[-1].set_xticks(month_week_positions)
axes1[-1].set_xticklabels(month_labels, fontsize=9.5, rotation=90, ha='center')
axes1[-1].set_xlabel('Time of Year', fontsize=11, fontweight='normal', labelpad=15)
plt.xlim(1, 52)

fig1.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5615, 0.93), ncol=2, frameon=True, facecolor='white', fontsize=10)
fig1.suptitle("Open Ocean Seasonality and Environmental Drivers", fontsize=12, fontweight='bold', x=0.5615, y=0.96)

fig1.tight_layout()
fig1.subplots_adjust(left = 0.193, right = 0.93, top=0.87, bottom=0.15, hspace=0.22)
fig1.savefig('C:/Users/25298423/PycharmProjects/FinalFigures/ocean_seasonality_shifts.png', dpi=1200)
plt.show()

# Want values for some of the plots

Going to get the mean values of the trends for the first year and last year of the time series, and calculate the difference

In [ ]:
df_coast_trends = df_stl_wrain.xs('trend', level=1, axis=1)

In [ ]:
coast_start_means = df_coast_trends[:52].mean()
coast_end_means = df_coast_trends[-52:].mean()

In [ ]:
coast_net_shift = coast_end_means - coast_start_means

In [ ]:
print(coast_start_means, coast_end_means, coast_net_shift)

In [ ]:
coast_start_means.index.name='parameter'
coast_start_means.name = 'mean_1997'
coast_end_means.index.name='parameter'
coast_end_means.name = 'mean_2025'
coast_net_shift.name = 'Net Change'
coast_change_means = pd.concat([coast_start_means, coast_end_means, coast_net_shift], axis=1)

order = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C'] #, 'lag_sumrain_7d', 'sumrain_7d', 'meanwind_7d', 'maxburst_7d']
coast_change_means = coast_change_means.reindex(order)
coast_change_means


In [ ]:
coast_change_means.to_excel('coastal_net_shift_1997_2025.xlsx')

In [ ]:
## Now for a five year version

In [ ]:
coast_start_means_big = df_coast_trends[:260].mean()
coast_end_means_big = df_coast_trends[-260:].mean()
coast_net_shift_big = coast_end_means_big - coast_start_means_big
coast_net_shift_big

In [ ]:
## Ocean 1 year version

In [ ]:
df_ocean_trends = df_ocean_stl.xs('trend', level=1, axis=1)

In [ ]:
ocean_start_means = df_ocean_trends[:52].mean().drop(index='sst')
ocean_end_means = df_ocean_trends[-52:].mean().drop(index='sst')

In [ ]:
ocean_start_means.index.name='parameter'
ocean_start_means.name = 'mean_1997'
ocean_end_means.index.name='parameter'
ocean_end_means.name = 'mean_2025'

In [ ]:
ocean_net_shift = ocean_end_means - ocean_start_means

In [ ]:
ocean_net_shift.name = 'Net Change'

In [ ]:
ocean_change_means = pd.concat([ocean_start_means, ocean_end_means, ocean_net_shift], axis=1)
ocean_change_means

In [ ]:
order = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C', 'lag_sumrain_7d', 'sumrain_7d', 'meanwind_7d','maxburst_7d']
ocean_change_means = ocean_change_means.reindex(order)
ocean_change_means

In [ ]:
ocean_change_means.to_excel('ocean_net_shift_1997_2025.xlsx')

In [ ]:
## 5 year ocean version

In [ ]:
ocean_start_means_big = df_ocean_trends[:260].mean().drop(index='sst')
ocean_end_means_big = df_ocean_trends[-260:].mean().drop(index='sst')
ocean_net_shift_big = ocean_end_means_big - ocean_start_means_big
ocean_net_shift_big

Now what years had the maximum values?

In [ ]:
## coastal max

In [ ]:
coast_max_date = df_coast_trends.idxmax().to_frame()
coast_max_value = df_coast_trends.max().to_frame()

In [ ]:
coast_max_date.rename_axis('parameter', inplace=True)
coast_max_date.rename(columns = {0:'max_date'}, inplace=True)
coast_max_value.rename_axis('parameter', inplace=True)
coast_max_value.rename(columns = {0:'max_value'}, inplace=True)

In [ ]:
coast_max = coast_max_date.join(coast_max_value, on='parameter', how='left')
order = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C', 'lag_sumrain_7d', 'sumrain_7d', 'meanwind_7d', 'maxburst_7d']
coast_max = coast_max.reindex(order)

In [ ]:
coast_max.to_excel('coastal_yearwithmax.xlsx')

In [ ]:
## now coastal min
coast_min_date = df_coast_trends.idxmin().to_frame()
coast_min_value = df_coast_trends.min().to_frame()
coast_min_date.rename_axis('parameter', inplace=True)
coast_min_date.rename(columns={0: 'min_date'}, inplace=True)
coast_min_value.rename_axis('parameter', inplace=True)
coast_min_value.rename(columns={0: 'min_value'}, inplace=True)
coast_min = coast_min_date.join(coast_min_value, on='parameter', how='left')
order = ['kd490', 'chl_a', 'spm', 'cdom', 'sst_C', 'lag_sumrain_7d', 'sumrain_7d', 'meanwind_7d', 'maxburst_7d']
coast_min = coast_min.reindex(order)
coast_min.to_excel('coastal_yearwithmin.xlsx')

In [ ]:

## now same for the ocean

In [ ]:
ocean_max_date = df_ocean_trends.idxmax().to_frame()
ocean_max_value = df_ocean_trends.max().to_frame()
ocean_max_date.rename_axis('parameter', inplace=True)
ocean_max_date.rename(columns={0: 'max_date'}, inplace=True)
ocean_max_value.rename_axis('parameter', inplace=True)
ocean_max_value.rename(columns={0: 'max_value'}, inplace=True)
ocean_max = ocean_max_date.join(ocean_max_value, on='parameter', how='left')
ocean_max

In [ ]:
ocean_max = ocean_max.reindex(order)

In [ ]:
ocean_max

In [ ]:
ocean_max.to_excel('ocean_yearwithmax.xlsx')

In [ ]:
## ocean min dates and values
ocean_min_date = df_ocean_trends.idxmin().to_frame()
ocean_min_value = df_ocean_trends.min().to_frame()
ocean_min_date.rename_axis('parameter', inplace=True)
ocean_min_date.rename(columns={0: 'min_date'}, inplace=True)
ocean_min_value.rename_axis('parameter', inplace=True)
ocean_min_value.rename(columns={0: 'min_value'}, inplace=True)
ocean_min = ocean_min_date.join(ocean_min_value, on='parameter', how='left')
ocean_min
ocean_min = ocean_min.reindex(order)
ocean_min
ocean_min.to_excel('ocean_yearwithmin.xlsx')

In [ ]:
ocean_min

**Now I want to see what week was highest for the seasonal trends**

** Coastal Early **

In [ ]:
df_coast_sea = df_stl_wrain.xs('seasonal', level=1, axis=1)

In [ ]:
def assign_custom_week(df):
    # Swap out the manual math for the exact ISO calendar week used in your plots
    df['custom_week'] = df.index.isocalendar().week
    return df

In [ ]:
coast_sea_early = assign_custom_week(df_coast_sea.loc['1997-09-07':'2002-09-07'].copy())
coast_sea_recent = assign_custom_week(df_coast_sea.loc['2020-09-07': '2025-09-07'].copy())


In [ ]:
date_lookup = (
    coast_sea_early.groupby('custom_week')
    .apply(lambda x: x.index[0].strftime('%d-%b'), include_groups=False)
    .to_dict()
)

In [ ]:

coast_prof_early = coast_sea_early.groupby('custom_week').mean()
coast_prof_recent = coast_sea_recent.groupby('custom_week').mean()

In [ ]:
coast_sea_maxdate_e = coast_prof_early.idxmax().to_frame(name='max_week')
coast_sea_maxdate_e.index.name = 'parameter'
coast_sea_maxvalue_e = coast_prof_early.max().to_frame(name='max_value')
coast_sea_maxvalue_e.index.name = 'parameter'

coast_sea_max_e = coast_sea_maxdate_e.join(coast_sea_maxvalue_e, on='parameter', how='left')


In [ ]:
coast_sea_max_e['max_date'] = coast_sea_max_e['max_week'].map(date_lookup)

coast_sea_max_e = coast_sea_max_e[['max_week','max_date', 'max_value']]
coast_sea_max_e

** Coastal Recent **

In [ ]:
date_lookup = (
    coast_sea_recent.groupby('custom_week')
    .apply(lambda x: x.index[0].strftime('%d-%b'), include_groups=False)
    .to_dict()
)
## custom week was already applied above
coast_prof_recent = coast_sea_recent.groupby('custom_week').mean()
coast_prof_recent = coast_sea_recent.groupby('custom_week').mean()

coast_sea_maxdate_r = coast_prof_recent.idxmax().to_frame(name='max_week')
coast_sea_maxdate_r.index.name = 'parameter'

coast_sea_maxvalue_r = coast_prof_recent.max().to_frame(name='max_value')
coast_sea_maxvalue_r.index.name = 'parameter'

coast_sea_max_r = coast_sea_maxdate_r.join(coast_sea_maxvalue_r, on='parameter', how='left')

coast_sea_max_r['max_date'] = coast_sea_max_r['max_week'].map(date_lookup)

coast_sea_max_r = coast_sea_max_r[['max_week', 'max_date', 'max_value']]
coast_sea_max_r

In [ ]:
## Function to get it all organized and output a xlxs file
import pandas as pd

def generate_phenology_table(df_early, df_recent, region_name="Region"):
    """
    Merges historical and modern phenology dataframes into a single master table,
    corrects for calendar year-wrap shifts, formats for publication, and
    exports directly to a distinct Excel file.
    """
    # 1. Clean and rename the historical era slice
    early_clean = df_early[['max_week', 'max_date', 'max_value']].copy()
    early_clean.columns = ['Hist_Week', 'Hist_Date', 'Hist_Max']

    # 2. Clean and rename the modern era slice
    recent_clean = df_recent[['max_week', 'max_date', 'max_value']].copy()
    recent_clean.columns = ['Mod_Week', 'Mod_Date', 'Mod_Max']

    # 3. Join them seamlessly together on the parameter index
    master_table = early_clean.join(recent_clean, how='inner')
    master_table.index.name = 'Parameter'

    desired_order = [
        'kd490', 'chl_a', 'spm', 'cdom', 'sst_C',
        'sumrain_7d', 'meanwind_7d', 'maxburst_7d'
    ]

    # Filter the desired order to only include parameters that actually exist in the dataframe
    existing_order = [p for p in desired_order if p in master_table.index]
    master_table = master_table.reindex(existing_order)

    hist_w = master_table['Hist_Week'].astype(int)
    mod_w = master_table['Mod_Week'].astype(int)
    # 4. Calculate raw week shifts and fix the New Year wrap-around boundary
    raw_shift = hist_w - mod_w
    corrected_shift = raw_shift.apply(lambda x: x - 52 if x > 26 else (x + 52 if x < -26 else x))

    # 5. Format the net shift column into clear text strings
    def format_shift(x):
        if x > 0:
            return f"+{int(x)} Wk (Earlier)"
        elif x < 0:
            return f"-{int(abs(x))} Wk (Later)"
        else:
            return "No Shift"

    master_table['Net Timing Shift'] = corrected_shift.apply(format_shift)

## percentage amplitude change
    pct_change = ((master_table['Mod_Max'] - master_table['Hist_Max']) / master_table['Hist_Max']) * 100

    def format_amplitude_shift(x):
        if x > 0:
            return f"+{x:.1f}%"
        elif x < 0:
            return f"{x:.1f}%"
        else:
            return "0.0%"

    master_table['Net Amp Change'] = pct_change.apply(format_amplitude_shift)

    # 6. Reorder columns to guarantee a logical left-to-right flow
    master_table = master_table[[
        'Hist_Week', 'Hist_Date', 'Hist_Max',
        'Mod_Week', 'Mod_Date', 'Mod_Max',
        'Net Timing Shift', 'Net Amp Change'
    ]]

    # 7. Generate a region-specific filename and export to Excel
    # .lower().replace(' ', '_') turns 'Offshore Ocean' into 'offshore_ocean'
    safe_region_name = region_name.lower().replace(' ', '_')
    filename = f"{safe_region_name}_seasonal_peak_shifts.xlsx"

    master_table.to_excel(filename)
    print(f"Success: Generated and saved '{filename}'")

    return master_table

In [ ]:
generate_phenology_table(coast_sea_max_e, coast_sea_max_r, region_name="coastal")

** Ocean Early **

In [ ]:
df_ocean_sea = df_ocean_stl.xs('seasonal', level=1, axis=1)


ocean_sea_early = assign_custom_week(df_ocean_sea.loc['1997-09-07':'2002-09-07'].copy())
ocean_sea_recent = assign_custom_week(df_ocean_sea.loc['2020-09-07': '2025-09-07'].copy())

date_lookup_early = (
    ocean_sea_early.groupby('custom_week')
    .apply(lambda x: x.index[0].strftime('%d-%b'), include_groups=False)
    .to_dict()
)
date_lookup_recent = (
    ocean_sea_recent.groupby('custom_week')
    .apply(lambda x: x.index[0].strftime('%d-%b'), include_groups=False)
    .to_dict()
)
ocean_prof_early = ocean_sea_early.groupby('custom_week').mean()
ocean_prof_recent = ocean_sea_recent.groupby('custom_week').mean()

In [ ]:


ocean_sea_maxdate_e = ocean_prof_early.idxmax().to_frame(name='max_week')
ocean_sea_maxdate_e.index.name = 'parameter'
ocean_sea_maxvalue_e = ocean_prof_early.max().to_frame(name='max_value')
ocean_sea_maxvalue_e.index.name = 'parameter'

ocean_sea_max_e = ocean_sea_maxdate_e.join(ocean_sea_maxvalue_e, on='parameter', how='left')

ocean_sea_max_e['max_date'] = ocean_sea_max_e['max_week'].map(date_lookup_early)

ocean_sea_max_e = ocean_sea_max_e[['max_week', 'max_date', 'max_value']]
ocean_sea_max_e

In [ ]:


ocean_sea_maxdate_r = ocean_prof_recent.idxmax().to_frame(name='max_week')
ocean_sea_maxdate_r.index.name = 'parameter'
ocean_sea_maxvalue_r = ocean_prof_recent.max().to_frame(name='max_value')
ocean_sea_maxvalue_r.index.name = 'parameter'

ocean_sea_max_r = ocean_sea_maxdate_r.join(ocean_sea_maxvalue_r, on='parameter', how='left')

ocean_sea_max_r['max_date'] = ocean_sea_max_r['max_week'].map(date_lookup_early)

ocean_sea_max_r = ocean_sea_max_r[['max_week', 'max_date', 'max_value']]
ocean_sea_max_r

In [ ]:
generate_phenology_table(ocean_sea_max_e, ocean_sea_max_r, region_name="ocean")